### Cell 1 — المسارات، المجتمعات المشتركة، وملفات Top-20

In [1]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 1 — Last.fm Recommendation Delivery: Paths and Common Communities
# =====================================================================================

from pathlib import Path
import os
import re
import json
import secrets
from datetime import datetime

import pandas as pd
from web3 import Web3


# =====================================================================================
# 1. Main Delivery Directory — Last.fm
# =====================================================================================

BASE_DELIVERY_DIR = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد"
)

CONTRACT_DIR = BASE_DELIVERY_DIR / "01_contract"
PYTHON_SCRIPTS_DIR = BASE_DELIVERY_DIR / "02_Python_scripts"
OUTPUTS_DIR = BASE_DELIVERY_DIR / "03_outputs"
COMMON_INPUTS_DIR = BASE_DELIVERY_DIR / "00_common_inputs"

CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
PYTHON_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
COMMON_INPUTS_DIR.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# 2. Previous Last.fm Announcement Stage Outputs
# =====================================================================================

LASTFM_ANNOUNCEMENT_DIR = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

ADDRESS_TO_USERID_PATH = (
    LASTFM_ANNOUNCEMENT_DIR / "data" / "blockchain" / "address_to_userid.csv"
)

COMMUNITY_MEMBERS_PATH = (
    LASTFM_ANNOUNCEMENT_DIR / "data" / "communities" / "community_members.csv"
)

OWNER_ADDRESS_PATH = (
    LASTFM_ANNOUNCEMENT_DIR / "data" / "blockchain" / "owner_address.txt"
)

ANNOUNCEMENT_LOGS_DIR = LASTFM_ANNOUNCEMENT_DIR / "outputs" / "logs"

LOAD_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "load_members_report.csv"
ANNOUNCE_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "announce_ak_report.csv"
CONFIRM_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "confirm_ak_report.csv"
FINALIZED_AK_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "finalized_ak_report.csv"


# =====================================================================================
# 3. Final Common Communities for Voting–Announcement–Delivery
# =====================================================================================

COMMON_COMMUNITIES = [
    28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506
]

EXCLUDED_ANNOUNCEMENT_COMMUNITIES = [
    5, 150, 163, 167, 265, 348
]


# =====================================================================================
# 4. Last.fm Top-20 Recommendation Files for Common Communities
# =====================================================================================

LASTFM_COMMON_TOP20_RECS_FILES = {
    28: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_28\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28.csv",

    30: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_30\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM30\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM30.csv",

    38: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_38\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM38\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM38.csv",

    40: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_40\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM40\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM40.csv",

    44: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_44\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM44\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM44.csv",

    60: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_60\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM60\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM60.csv",

    99: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_99\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM99\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM99.csv",

    104: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_104\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM104\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM104.csv",

    107: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_107\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM107\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM107.csv",

    114: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_114\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM114\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM114.csv",

    204: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_204\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM204\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM204.csv",

    211: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_211\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM211\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM211.csv",

    506: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_506\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM506\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM506.csv",
}

RECOMMENDATION_FILES = [
    Path(p)
    for _, p in sorted(LASTFM_COMMON_TOP20_RECS_FILES.items())
]


# =====================================================================================
# 5. Check Required Files
# =====================================================================================

required_files = {
    "address_to_userid.csv": ADDRESS_TO_USERID_PATH,
    "community_members.csv": COMMUNITY_MEMBERS_PATH,
    "owner_address.txt": OWNER_ADDRESS_PATH,
    "load_members_report.csv": LOAD_REPORT_PATH,
    "announce_ak_report.csv": ANNOUNCE_REPORT_PATH,
    "confirm_ak_report.csv": CONFIRM_REPORT_PATH,
    "finalized_ak_report.csv": FINALIZED_AK_REPORT_PATH,
}

print("=" * 100)
print("Cell 1 — Last.fm Recommendation Delivery Paths")
print("=" * 100)

print("BASE_DELIVERY_DIR:", BASE_DELIVERY_DIR)
print("CONTRACT_DIR:", CONTRACT_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("COMMON_INPUTS_DIR:", COMMON_INPUTS_DIR)

print("\nChecking required announcement-stage files:")
missing_files = []

for name, path in required_files.items():
    exists = path.exists()
    print(f"{name}: {exists} => {path}")
    if not exists:
        missing_files.append(name)

if missing_files:
    raise FileNotFoundError(f"Missing required announcement-stage files: {missing_files}")

print("\nChecking recommendation files:")
missing_rec_files = []

for cid, p_str in sorted(LASTFM_COMMON_TOP20_RECS_FILES.items()):
    p = Path(p_str)
    exists = p.exists()
    print(f"community={cid} | exists={exists} | {p}")
    if not exists:
        missing_rec_files.append((cid, p))

if missing_rec_files:
    print("\n[WARNING] Some recommendation files are missing.")
    for cid, p in missing_rec_files:
        print(f"Missing: community={cid} => {p}")
else:
    print("\n[OK] All recommendation files exist.")

print("\n[OK] Cell 1 completed.")

Cell 1 — Last.fm Recommendation Delivery Paths
BASE_DELIVERY_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد
CONTRACT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract
OUTPUTS_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs
COMMON_INPUTS_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs

Checking required announcement-stage files:
address_to_userid.csv: True => D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\blockchain\address_to_userid.csv
community_members.csv: True => D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\communities\community_members.csv
owner_address.txt: True => D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\blockchain\owner_address.txt
load_members_report.csv: True => D:\PHD\بابل\2024\Research Reco\الجز

### فائدة Cell 2 ببساطة

هذه الخلية لا تتعامل مع العقد الذكي ولا ترسل معاملات. فائدتها تنظيمية ومنهجية.

هي تقوم بثلاث وظائف مهمة:

1. تأخذ نتائج الإعلان الكاملة التي كانت لـ 19 مجتمعًا.
2. تستخرج منها فقط المجتمعات المشتركة الـ13 المعتمدة في التصويت–الإعلان–الإيصال.
3. تنشئ ملف Manifest يربط كل مجتمع مع A_k النهائي وملف توصيات Top-20 الخاص به.

يعني هي تجهز الملفات التي سيعتمد عليها كود الإيصال لاحقًا حتى لا نخلط بين:

مجتمعات الإعلان الكاملة = 19
والمجتمعات المشتركة النهائية = 13
### Cell 2 — Create Common-Community Manifest and Filtered Announcement Reports

In [2]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 2 — Create Common-Community Manifest and Filtered Announcement Reports
# =====================================================================================
#
# Purpose:
#   هذه الخلية تجهز ملفات الإدخال الخاصة بمرحلة إيصال التوصيات.
#
# Why this cell is important:
#   في مرحلة الإعلان، تم تثبيت A_k لعدد 19 مجتمعًا.
#   لكن في مرحلة إيصال التوصيات، سنعمل فقط على المجتمعات المشتركة التي تمتلك:
#
#       1. نتيجة تصويت A_k
#       2. إعلان وتثبيت A_k
#       3. ملف توصيات Top-20
#
#   لذلك نحتاج إلى تصفية نتائج الإعلان لتشمل المجتمعات المشتركة فقط:
#
#       [28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506]
#
# What this cell generates:
#   1. load_members_report_common_communities.csv
#   2. announce_ak_report_common_communities.csv
#   3. confirm_ak_report_common_communities.csv
#   4. finalized_ak_report_common_communities.csv
#   5. lastfm_excluded_announcement_communities.csv
#   6. lastfm_common_communities_manifest.csv
#
# Main output:
#   lastfm_common_communities_manifest.csv
#
# This manifest will be used later to connect:
#
#       Voting  →  Announcement  →  Recommendation Delivery
#
# =====================================================================================


print("=" * 100)
print("Cell 2 — Create Common-Community Input Files")
print("=" * 100)


# =====================================================================================
# 1. Read full announcement-stage reports
# =====================================================================================

load_df = pd.read_csv(LOAD_REPORT_PATH)
announce_df = pd.read_csv(ANNOUNCE_REPORT_PATH)
confirm_df = pd.read_csv(CONFIRM_REPORT_PATH)
finalized_df = pd.read_csv(FINALIZED_AK_REPORT_PATH)


# =====================================================================================
# 2. Make sure community_id is numeric in all reports
# =====================================================================================

for df in [load_df, announce_df, confirm_df, finalized_df]:
    df["community_id"] = df["community_id"].astype(int)


# =====================================================================================
# 3. Keep only the final common communities
# =====================================================================================

load_common_df = load_df[
    load_df["community_id"].isin(COMMON_COMMUNITIES)
].copy()

announce_common_df = announce_df[
    announce_df["community_id"].isin(COMMON_COMMUNITIES)
].copy()

confirm_common_df = confirm_df[
    confirm_df["community_id"].isin(COMMON_COMMUNITIES)
].copy()

finalized_common_df = finalized_df[
    finalized_df["community_id"].isin(COMMON_COMMUNITIES)
].copy()


# =====================================================================================
# 4. Save excluded announcement communities separately
# =====================================================================================
#
# These communities were finalized in the announcement stage,
# but they are not included in the integrated delivery analysis
# because Top-20 recommendation files are not available for them.
# =====================================================================================

excluded_finalized_df = finalized_df[
    finalized_df["community_id"].isin(EXCLUDED_ANNOUNCEMENT_COMMUNITIES)
].copy()


# =====================================================================================
# 5. Define output paths
# =====================================================================================

LOAD_COMMON_PATH = COMMON_INPUTS_DIR / "load_members_report_common_communities.csv"

ANNOUNCE_COMMON_PATH = COMMON_INPUTS_DIR / "announce_ak_report_common_communities.csv"

CONFIRM_COMMON_PATH = COMMON_INPUTS_DIR / "confirm_ak_report_common_communities.csv"

FINALIZED_COMMON_PATH = COMMON_INPUTS_DIR / "finalized_ak_report_common_communities.csv"

EXCLUDED_COMMUNITIES_PATH = COMMON_INPUTS_DIR / "lastfm_excluded_announcement_communities.csv"

COMMON_MANIFEST_PATH = COMMON_INPUTS_DIR / "lastfm_common_communities_manifest.csv"


# =====================================================================================
# 6. Save filtered common-community reports
# =====================================================================================

load_common_df.to_csv(
    LOAD_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

announce_common_df.to_csv(
    ANNOUNCE_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

confirm_common_df.to_csv(
    CONFIRM_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

finalized_common_df.to_csv(
    FINALIZED_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

excluded_finalized_df.to_csv(
    EXCLUDED_COMMUNITIES_PATH,
    index=False,
    encoding="utf-8-sig"
)


# =====================================================================================
# 7. Build the common-community manifest
# =====================================================================================
#
# The manifest is the most important file in this cell.
#
# It links each common community with:
#   - finalized A_k
#   - confirmation count
#   - required confirmations
#   - community size, if available
#   - Top-20 recommendation file path
#   - whether the community is included in delivery
#
# =====================================================================================

manifest_rows = []

for cid in COMMON_COMMUNITIES:

    f_row = finalized_common_df[
        finalized_common_df["community_id"] == cid
    ]

    if f_row.empty:
        raise RuntimeError(
            f"Community {cid} not found in finalized_ak_report.csv"
        )

    f_row = f_row.iloc[0]


    # -------------------------------------------------------------------------
    # Read finalized A_k
    # -------------------------------------------------------------------------

    if "finalized_Ak" in f_row.index:
        finalized_ak = int(f_row["finalized_Ak"])

    elif "expected_Ak" in f_row.index:
        finalized_ak = int(f_row["expected_Ak"])

    else:
        raise RuntimeError(
            "Cannot find finalized_Ak or expected_Ak column."
        )


    # -------------------------------------------------------------------------
    # Read required confirmations
    # -------------------------------------------------------------------------

    if "required_confirmations" in f_row.index:
        required = int(f_row["required_confirmations"])

    elif "required_count" in f_row.index:
        required = int(f_row["required_count"])

    else:
        required = None


    # -------------------------------------------------------------------------
    # Read confirmation count
    # -------------------------------------------------------------------------

    if "confirm_count" in f_row.index:
        confirm_count = int(f_row["confirm_count"])
    else:
        confirm_count = None


    # -------------------------------------------------------------------------
    # Try to read community size from the loading report
    # -------------------------------------------------------------------------

    community_size = None

    l_row = load_common_df[
        load_common_df["community_id"] == cid
    ]

    possible_size_cols = [
        "onchain_size_after",
        "member_count_onchain",
        "expected_members",
        "expected_size",
        "community_size"
    ]

    if not l_row.empty:
        for c in possible_size_cols:
            if c in l_row.columns:
                community_size = int(l_row.iloc[0][c])
                break


    # -------------------------------------------------------------------------
    # Link community with its Top-20 recommendation file
    # -------------------------------------------------------------------------

    recommendation_file = LASTFM_COMMON_TOP20_RECS_FILES.get(cid)

    recommendation_file_exists = Path(
        recommendation_file
    ).exists()


    # -------------------------------------------------------------------------
    # Read validation status
    # -------------------------------------------------------------------------

    validation_status = "valid_finalized"

    if "validation_status" in f_row.index:
        validation_status = str(f_row["validation_status"])


    # -------------------------------------------------------------------------
    # Add one manifest row
    # -------------------------------------------------------------------------

    manifest_rows.append(
        {
            "dataset": "Last.fm",
            "community_id": int(cid),
            "finalized_Ak": finalized_ak,
            "community_size": community_size,
            "confirm_count": confirm_count,
            "required_confirmations": required,
            "validation_status": validation_status,
            "recommendation_file": str(recommendation_file),
            "recommendation_file_exists": bool(recommendation_file_exists),
            "included_in_delivery": bool(recommendation_file_exists),
        }
    )


# =====================================================================================
# 8. Save manifest
# =====================================================================================

manifest_df = pd.DataFrame(manifest_rows)

manifest_df.to_csv(
    COMMON_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)


# =====================================================================================
# 9. Print output summary
# =====================================================================================

print("[OK] Common-community files created:")
print("1.", LOAD_COMMON_PATH)
print("2.", ANNOUNCE_COMMON_PATH)
print("3.", CONFIRM_COMMON_PATH)
print("4.", FINALIZED_COMMON_PATH)
print("5.", EXCLUDED_COMMUNITIES_PATH)
print("6.", COMMON_MANIFEST_PATH)

print("\n[COMMON MANIFEST]")
print(manifest_df.to_string(index=False))


# =====================================================================================
# 10. Final validation
# =====================================================================================

if not manifest_df["recommendation_file_exists"].all():
    raise FileNotFoundError(
        "Some common recommendation files do not exist. Check manifest."
    )

if len(manifest_df) != len(COMMON_COMMUNITIES):
    raise RuntimeError(
        f"Manifest size mismatch. Expected {len(COMMON_COMMUNITIES)}, "
        f"but got {len(manifest_df)}"
    )

invalid_status = manifest_df[
    manifest_df["validation_status"] != "valid_finalized"
]

if not invalid_status.empty:
    raise RuntimeError(
        "Some common communities are not valid_finalized:\n"
        + invalid_status.to_string(index=False)
    )

print("\n[FINAL CHECK]")
print("All common communities are valid_finalized and have Top-20 recommendation files.")

print("\n[OK] Cell 2 completed.")

Cell 2 — Create Common-Community Input Files
[OK] Common-community files created:
1. D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs\load_members_report_common_communities.csv
2. D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs\announce_ak_report_common_communities.csv
3. D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs\confirm_ak_report_common_communities.csv
4. D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs\finalized_ak_report_common_communities.csv
5. D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs\lastfm_excluded_announcement_communities.csv
6. D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\00_common_inputs\lastfm_common_communities_manifest.csv

[COMMON MANIFEST]
dataset  communi

### هذه هي Cell 3. 
فائدتها أنها تكتب كود العقد الذكي الخاص بمرحلة إيصال التوصيات داخل ملف:

01_contract\RecommendationDelivery.sol

هذا العقد سيُستخدم لاحقًا في تخزين Hash التوصيات فقط، وليس التوصيات الكاملة.

Cell 3 — Write RecommendationDelivery Solidity Smart Contrac

In [3]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 3 — Write RecommendationDelivery Solidity Smart Contract
# =====================================================================================
#
# Purpose:
#   هذه الخلية تكتب كود العقد الذكي الخاص بمرحلة إيصال التوصيات.
#
# What the contract does:
#   1. يسجل العقدة الموثوقة A_k لكل مجتمع.
#   2. يسجل أعضاء كل مجتمع داخل العقد.
#   3. يسمح فقط لـ A_k الخاصة بالمجتمع بتخزين Hash التوصيات.
#   4. يمنع تخزين Hash لمستخدم غير عضو في المجتمع.
#   5. يخزن Hash فقط وليس قائمة التوصيات الكاملة.
#
# Why this is important:
#   التوصيات الكاملة تبقى خارج البلوكشين داخل Payload JSON.
#   أما البلوكشين فيخزن بصمة Hash فقط للتحقق من سلامة التوصيات لاحقًا.
#
# Output:
#   D:\...\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract\RecommendationDelivery.sol
# =====================================================================================


SOLIDITY_CODE = r'''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/*
    RecommendationDelivery

    This contract supports the recommendation delivery stage.

    It does not store full recommendations on-chain.
    Instead, it stores only the hash of each recommendation payload.

    Main functions:
    - Register the trusted authorized node A_k for each community.
    - Register community members.
    - Allow only A_k to store recommendation hashes.
    - Reject hashes for users who are not registered community members.
    - Verify whether an off-chain payload matches the on-chain hash.
*/

contract RecommendationDelivery {

    address public owner;

    struct RecommendationRecord {
        bytes32 recommendationHash;
        address issuer;
        uint256 timestamp;
        bool exists;
    }

    // communityId => authorized node A_k address
    mapping(uint256 => address) public authorizedNode;

    // communityId => userAddress => membership status
    mapping(uint256 => mapping(address => bool)) public isCommunityMember;

    // communityId => number of registered members
    mapping(uint256 => uint256) public communityMemberCount;

    /*
        records[communityId][user][version][topN] = RecommendationRecord
    */
    mapping(uint256 => mapping(address => mapping(uint256 => mapping(uint256 => RecommendationRecord)))) private records;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Only owner can perform this action");
        _;
    }

    modifier onlyAuthorizedNode(uint256 communityId) {
        require(
            msg.sender == authorizedNode[communityId],
            "Only authorized node can store recommendation hashes"
        );
        _;
    }

    /*
        Set the trusted authorized node A_k for a community.
        This is executed by the owner after reading the finalized announcement results.
    */
    function setAuthorizedNode(
        uint256 communityId,
        address node
    ) external onlyOwner {
        require(node != address(0), "Invalid authorized node address");
        authorizedNode[communityId] = node;
    }

    /*
        Register a batch of community members.
        The contract uses user addresses, not raw dataset user IDs.
    */
    function registerCommunityMembers(
        uint256 communityId,
        address[] calldata members
    ) external onlyOwner {
        for (uint256 i = 0; i < members.length; i++) {
            address member = members[i];
            require(member != address(0), "Invalid member address");

            if (!isCommunityMember[communityId][member]) {
                isCommunityMember[communityId][member] = true;
                communityMemberCount[communityId] += 1;
            }
        }
    }

    /*
        Check whether an address is registered as a member of a community.
    */
    function checkCommunityMember(
        uint256 communityId,
        address member
    ) external view returns (bool) {
        return isCommunityMember[communityId][member];
    }

    /*
        Store the recommendation hash only.

        Acceptance conditions:
        1. msg.sender must be A_k of the corresponding community.
        2. target user must be a registered member of that community.
        3. the same key must not already exist:
           communityId + user + version + topN.
    */
    function storeRecommendationHash(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN,
        bytes32 recommendationHash
    ) external onlyAuthorizedNode(communityId) {
        require(user != address(0), "Invalid user address");
        require(topN > 0, "Invalid topN value");
        require(recommendationHash != bytes32(0), "Invalid recommendation hash");

        require(
            isCommunityMember[communityId][user],
            "Target user is not a registered member of this community"
        );

        require(
            records[communityId][user][version][topN].exists == false,
            "Recommendation hash already exists for this key"
        );

        records[communityId][user][version][topN] = RecommendationRecord({
            recommendationHash: recommendationHash,
            issuer: msg.sender,
            timestamp: block.timestamp,
            exists: true
        });
    }

    /*
        Check whether a recommendation record already exists.
    */
    function recordExists(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN
    ) external view returns (bool) {
        return records[communityId][user][version][topN].exists;
    }

    /*
        Get the stored recommendation hash.
    */
    function getRecommendationHash(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN
    ) external view returns (bytes32) {
        require(
            records[communityId][user][version][topN].exists,
            "Recommendation record not found"
        );

        return records[communityId][user][version][topN].recommendationHash;
    }

    /*
        Verify whether the locally computed hash matches the stored on-chain hash.

        This verifies integrity only.
        It does not evaluate recommendation quality.
    */
    function verifyRecommendationHash(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN,
        bytes32 receivedHash
    ) external view returns (bool) {
        if (!records[communityId][user][version][topN].exists) {
            return false;
        }

        return records[communityId][user][version][topN].recommendationHash == receivedHash;
    }

    /*
        Get the full on-chain record metadata for audit and validation.
    */
    function getRecommendationRecord(
        uint256 communityId,
        address user,
        uint256 version,
        uint256 topN
    ) external view returns (
        bytes32 recommendationHash,
        address issuer,
        uint256 timestamp,
        bool exists
    ) {
        RecommendationRecord memory record = records[communityId][user][version][topN];

        return (
            record.recommendationHash,
            record.issuer,
            record.timestamp,
            record.exists
        );
    }
}
'''


SOLIDITY_FILE = CONTRACT_DIR / "RecommendationDelivery.sol"

SOLIDITY_FILE.write_text(
    SOLIDITY_CODE,
    encoding="utf-8"
)

print("=" * 100)
print("Cell 3 — Write RecommendationDelivery Solidity Smart Contract")
print("=" * 100)

print("[OK] Smart contract source saved to:")
print(SOLIDITY_FILE)

print("\n[CONTRACT SUMMARY]")
print("Contract name: RecommendationDelivery")
print("Stores: recommendation hash only")
print("Does not store: full recommendation payload")
print("Authorization: only finalized A_k can store hashes for its community")
print("Membership rule: target user must be a registered community member")

print("\n[OK] Cell 3 completed.")

Cell 3 — Write RecommendationDelivery Solidity Smart Contract
[OK] Smart contract source saved to:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract\RecommendationDelivery.sol

[CONTRACT SUMMARY]
Contract name: RecommendationDelivery
Stores: recommendation hash only
Does not store: full recommendation payload
Authorization: only finalized A_k can store hashes for its community
Membership rule: target user must be a registered community member

[OK] Cell 3 completed.


### نعم، ننتقل إلى Cell 4 — Connect to Ganache.

فائدة Cell 4 ببساطة

هذه الخلية لا تنشر العقد بعد، فقط تتأكد أن:

1. Ganache شغال على http://127.0.0.1:7545
2. يوجد حسابات كافية
3. حساب owner الحالي في Ganache هو نفس owner المستخدم في مرحلة إعلان A_k

إذا ظهر تطابق بين:

Owner Account from Ganache
Owner Account from Announcement

فهذا يعني أن بيئة Ganache صحيحة، ونقدر نكمل إلى Cell 5 لنشر عقد إيصال التوصيات.

### Cell 4 — Connect to Ganache

In [4]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 4 — Connect to Ganache
# =====================================================================================
#
# Purpose:
#   هذه الخلية تتصل بـ Ganache وتتحقق أن نفس الحسابات المستخدمة في مرحلة الإعلان
#   ما زالت متاحة في مرحلة إيصال التوصيات.
#
# Why this is important:
#   في مرحلة الإيصال، العقدة الموثوقة A_k سترسل معاملات تخزين Hash التوصيات.
#   لذلك يجب أن تكون عناوين A_k موجودة ضمن حسابات Ganache الحالية.
#
# Main validation:
#   نقارن owner account الحالي في Ganache مع owner address المخزن من مرحلة الإعلان.
#
# If they match:
#   نكمل إلى نشر عقد RecommendationDelivery في Cell 5.
#
# If they do not match:
#   يعني أن Ganache تغير أو أُعيد تشغيله بحسابات مختلفة.
# =====================================================================================


RPC_URL = "http://127.0.0.1:7545"

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise RuntimeError(
        "Cannot connect to Ganache. تأكد أن Ganache شغال على http://127.0.0.1:7545"
    )

accounts = w3.eth.accounts

if len(accounts) < 2:
    raise RuntimeError(
        "Ganache must provide at least two accounts."
    )

OWNER_ACCOUNT = Web3.to_checksum_address(accounts[0])

owner_from_announcement = Web3.to_checksum_address(
    OWNER_ADDRESS_PATH.read_text(encoding="utf-8").strip()
)

print("=" * 100)
print("Cell 4 — Connect to Ganache")
print("=" * 100)

print("Connected to Ganache:", w3.is_connected())
print("Chain ID:", w3.eth.chain_id)
print("Number of accounts:", len(accounts))
print("Owner Account from Ganache:", OWNER_ACCOUNT)
print("Owner Account from Announcement:", owner_from_announcement)

if OWNER_ACCOUNT.lower() != owner_from_announcement.lower():
    raise RuntimeError(
        "Owner account mismatch. يبدو أن Ganache تغير أو أُعيد تشغيله بحسابات مختلفة."
    )

print("\nFirst Ganache accounts:")
for i, acc in enumerate(accounts[:10]):
    print(i, acc)

print("\n[FINAL CHECK]")
print("Owner account matches the announcement-stage owner.")
print("Ganache environment is consistent with the previous Last.fm announcement stage.")

print("\n[OK] Cell 4 completed.")

Cell 4 — Connect to Ganache
Connected to Ganache: True
Chain ID: 1337
Number of accounts: 850
Owner Account from Ganache: 0xf24B340783Aa1bbE02A67eA48557B3aD2b7f3e92
Owner Account from Announcement: 0xf24B340783Aa1bbE02A67eA48557B3aD2b7f3e92

First Ganache accounts:
0 0xf24B340783Aa1bbE02A67eA48557B3aD2b7f3e92
1 0x8824B1EF12A30f9D5a2e9319588EB8779a6997b6
2 0x0cc3Cf53430059e546Eb461ddA175b0DCdc97fAb
3 0xc8B65A14DE27749A8EF48D61a88d5885812f35F8
4 0xd8a8C65080b300028D3D95D56f4dFff7bECE8fa9
5 0xD9348F45EfcB640aCB7e1F2f2aD966825416cfaF
6 0x5b1f0690c78e933ef302CBaCb346e7bd08384bD1
7 0xAE009A5CE582ef30A04f2E153Bbc9bA9b1E17D25
8 0xC44Ce4395F1B717dc105EA18AdeEbADF271a759C
9 0x933cb317eb006b58455BCd3dAbD5C90c00F8242B

[FINAL CHECK]
Owner account matches the announcement-stage owner.
Ganache environment is consistent with the previous Last.fm announcement stage.

[OK] Cell 4 completed.


### فائدة Cell 5 ببساطة

هذه الخلية تقوم بثلاث عمليات مهمة:

1. تعمل Compile للعقد RecommendationDelivery.sol
2. تنشر العقد على Ganache الحالي
3. تحفظ ABI + عنوان العقد + سجل النشر

بعد نجاحها سيصبح عندنا عقد إيصال التوصيات منشور فعليًا على Ganache، وسيتم حفظ عنوانه في:

D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract\contract_address.txt
Cell 5 — Compile and Deploy RecommendationDelivery Smart Contract

In [5]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 5 — Compile and Deploy RecommendationDelivery Smart Contract
# =====================================================================================
#
# Purpose:
#   هذه الخلية تعمل Compile ثم Deploy لعقد RecommendationDelivery على Ganache.
#
# What this cell does:
#   1. تستخدم Solidity compiler version 0.8.20.
#   2. تقرأ ملف العقد من:
#        01_contract\RecommendationDelivery.sol
#   3. تعمل Compile للعقد.
#   4. تنشر العقد على Ganache باستخدام OWNER_ACCOUNT.
#   5. تحفظ ABI في:
#        01_contract\RecommendationDelivery_abi.json
#   6. تحفظ عنوان العقد المنشور في:
#        01_contract\contract_address.txt
#   7. تحفظ Deployment Log في:
#        03_outputs\04_smart_contract_deployment_log.json
#
# Important:
#   هذا العقد مختلف عن عقد إعلان A_k السابق.
#   عقد الإعلان السابق يبقى كما هو.
#   هذا العقد الجديد خاص بإيصال التوصيات وتخزين Hash فقط.
# =====================================================================================


from solcx import compile_standard, install_solc, set_solc_version


# =====================================================================================
# 1. Solidity compiler configuration
# =====================================================================================

SOLC_VERSION = "0.8.20"

ABI_OUTPUT_PATH = CONTRACT_DIR / "RecommendationDelivery_abi.json"
ADDRESS_OUTPUT_PATH = CONTRACT_DIR / "contract_address.txt"
DEPLOYMENT_LOG_PATH = OUTPUTS_DIR / "04_smart_contract_deployment_log.json"

print("=" * 100)
print("Cell 5 — Compile and Deploy RecommendationDelivery Smart Contract")
print("=" * 100)

print("Installing / selecting Solidity compiler:", SOLC_VERSION)

install_solc(SOLC_VERSION)
set_solc_version(SOLC_VERSION)


# =====================================================================================
# 2. Read Solidity source code
# =====================================================================================

if not SOLIDITY_FILE.exists():
    raise FileNotFoundError(f"Solidity file not found: {SOLIDITY_FILE}")

source_code = SOLIDITY_FILE.read_text(encoding="utf-8")

print("\n[OK] Solidity source loaded from:")
print(SOLIDITY_FILE)


# =====================================================================================
# 3. Compile Solidity contract
# =====================================================================================

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {
            "RecommendationDelivery.sol": {
                "content": source_code
            }
        },
        "settings": {
            "optimizer": {
                "enabled": True,
                "runs": 200
            },
            "outputSelection": {
                "*": {
                    "*": [
                        "abi",
                        "evm.bytecode"
                    ]
                }
            }
        }
    }
)

contract_data = compiled["contracts"]["RecommendationDelivery.sol"]["RecommendationDelivery"]

contract_abi = contract_data["abi"]
contract_bytecode = contract_data["evm"]["bytecode"]["object"]

if not contract_bytecode:
    raise RuntimeError("Compiled bytecode is empty. Contract compilation failed.")

print("\n[OK] Compilation completed successfully.")
print("Contract name: RecommendationDelivery")


# =====================================================================================
# 4. Deploy contract on Ganache
# =====================================================================================

RecommendationDelivery = w3.eth.contract(
    abi=contract_abi,
    bytecode=contract_bytecode
)

print("\nDeploying RecommendationDelivery contract...")

deploy_tx_hash = RecommendationDelivery.constructor().transact(
    {
        "from": OWNER_ACCOUNT,
        "gas": 7_000_000
    }
)

deploy_receipt = w3.eth.wait_for_transaction_receipt(deploy_tx_hash)

if deploy_receipt.status != 1:
    raise RuntimeError("Contract deployment failed.")

contract_address = Web3.to_checksum_address(deploy_receipt.contractAddress)

print("\n[OK] Contract deployed successfully.")
print("Contract address:", contract_address)
print("Deployment tx:", deploy_tx_hash.hex())


# =====================================================================================
# 5. Save ABI and contract address
# =====================================================================================

ABI_OUTPUT_PATH.write_text(
    json.dumps(contract_abi, indent=2),
    encoding="utf-8"
)

ADDRESS_OUTPUT_PATH.write_text(
    contract_address,
    encoding="utf-8"
)

print("\n[OK] ABI saved to:")
print(ABI_OUTPUT_PATH)

print("\n[OK] Contract address saved to:")
print(ADDRESS_OUTPUT_PATH)


# =====================================================================================
# 6. Save deployment log
# =====================================================================================

deployment_log = {
    "deployment_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "rpc_url": RPC_URL,
    "chain_id": w3.eth.chain_id,
    "contract_name": "RecommendationDelivery",
    "contract_address": contract_address,
    "deployment_tx_hash": deploy_tx_hash.hex(),
    "owner_account": OWNER_ACCOUNT,
    "solidity_file": str(SOLIDITY_FILE),
    "abi_path": str(ABI_OUTPUT_PATH),
    "contract_address_path": str(ADDRESS_OUTPUT_PATH),
    "solc_version": SOLC_VERSION,
    "ganache_accounts_count": len(accounts),
    "common_communities_count": len(COMMON_COMMUNITIES),
    "common_communities": COMMON_COMMUNITIES,
    "note": "This contract stores recommendation hashes only for common Last.fm communities."
}

DEPLOYMENT_LOG_PATH.write_text(
    json.dumps(deployment_log, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[OK] Deployment log saved to:")
print(DEPLOYMENT_LOG_PATH)


# =====================================================================================
# 7. Final summary
# =====================================================================================

print("\n" + "=" * 100)
print("[FINAL SUMMARY]")
print("Dataset:", "Last.fm")
print("Stage:", "Recommendation Delivery")
print("Contract:", "RecommendationDelivery")
print("Contract address:", contract_address)
print("Chain ID:", w3.eth.chain_id)
print("Owner:", OWNER_ACCOUNT)
print("Common communities:", COMMON_COMMUNITIES)
print("=" * 100)

print("\n[OK] Cell 5 completed.")

Cell 5 — Compile and Deploy RecommendationDelivery Smart Contract
Installing / selecting Solidity compiler: 0.8.20

[OK] Solidity source loaded from:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract\RecommendationDelivery.sol

[OK] Compilation completed successfully.
Contract name: RecommendationDelivery

Deploying RecommendationDelivery contract...

[OK] Contract deployed successfully.
Contract address: 0x5071DE85Ef0917C7aAf5943Fe98e2806A0C637bd
Deployment tx: 140476903bd7add95303c290463c47b22c2d0bfc97a060f560d8e2a72df3e473

[OK] ABI saved to:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract\RecommendationDelivery_abi.json

[OK] Contract address saved to:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\01_contract\contract_address.txt

[OK] Deployment log saved to:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_ou

### فائدة Cell 6 ببساطة

هذه الخلية لا ترسل معاملات إلى العقد. فائدتها أنها تجهّز الربط بين:

1. عقد RecommendationDelivery المنشور في Cell 5
2. ملف manifest للمجتمعات المشتركة
3. عناوين المستخدمين من address_to_userid.csv
4. قيم A_k النهائية للمجتمعات المشتركة

يعني بعد هذه الخلية يصبح الكود يعرف:

كل مجتمع من الـ13
من هو A_k النهائي له
ما هو عنوان A_k في Ganache
هل هذا العنوان مفتوح ضمن حسابات Ganache

وهذا ضروري قبل تسجيل A_k والأعضاء داخل عقد الإيصال في Cell 9.

### Cell 6 — Load Contract and Common Last.fm Data

In [6]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 6 — Load RecommendationDelivery Contract and Common Last.fm Data
# =====================================================================================
#
# Purpose:
#   هذه الخلية تحمّل عقد RecommendationDelivery المنشور في Cell 5،
#   ثم تحمّل بيانات المجتمعات المشتركة اللازمة لمرحلة إيصال التوصيات.
#
# What this cell does:
#   1. تقرأ ABI الخاص بالعقد.
#   2. تقرأ عنوان العقد المنشور من contract_address.txt.
#   3. تنشئ كائن contract للتعامل مع العقد من Python.
#   4. تقرأ address_to_userid.csv لربط user_id بعنوان blockchain.
#   5. تقرأ community_members.csv لمعرفة أعضاء كل مجتمع.
#   6. تقرأ lastfm_common_communities_manifest.csv لمعرفة A_k النهائي وملفات Top-20.
#   7. تتحقق أن عناوين A_k موجودة ضمن حسابات Ganache المفتوحة.
#
# Output:
#   03_outputs\05_lastfm_trusted_nodes_common_from_finalized_announcement.csv
#
# =====================================================================================


print("=" * 100)
print("Cell 6 — Load RecommendationDelivery Contract and Common Last.fm Data")
print("=" * 100)


# =====================================================================================
# 1. Load deployed contract ABI and address
# =====================================================================================

if not ABI_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"ABI file not found: {ABI_OUTPUT_PATH}")

if not ADDRESS_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Contract address file not found: {ADDRESS_OUTPUT_PATH}")

with open(ABI_OUTPUT_PATH, "r", encoding="utf-8") as f:
    loaded_abi = json.load(f)

loaded_contract_address = Web3.to_checksum_address(
    ADDRESS_OUTPUT_PATH.read_text(encoding="utf-8").strip()
)

contract = w3.eth.contract(
    address=loaded_contract_address,
    abi=loaded_abi
)

print("[OK] Loaded RecommendationDelivery contract at:")
print(loaded_contract_address)


# =====================================================================================
# 2. Verify contract owner
# =====================================================================================

contract_owner = Web3.to_checksum_address(
    contract.functions.owner().call()
)

print("\nContract owner:", contract_owner)
print("Expected owner:", OWNER_ACCOUNT)

if contract_owner.lower() != OWNER_ACCOUNT.lower():
    raise RuntimeError(
        "Contract owner does not match OWNER_ACCOUNT. تحقق من العقد المنشور أو حساب Ganache."
    )

print("[OK] Contract owner matches OWNER_ACCOUNT.")


# =====================================================================================
# 3. Load common input files
# =====================================================================================

if not ADDRESS_TO_USERID_PATH.exists():
    raise FileNotFoundError(f"address_to_userid.csv not found: {ADDRESS_TO_USERID_PATH}")

if not COMMUNITY_MEMBERS_PATH.exists():
    raise FileNotFoundError(f"community_members.csv not found: {COMMUNITY_MEMBERS_PATH}")

if not COMMON_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Common manifest not found: {COMMON_MANIFEST_PATH}")

if not FINALIZED_COMMON_PATH.exists():
    raise FileNotFoundError(f"Finalized common report not found: {FINALIZED_COMMON_PATH}")


address_df = pd.read_csv(ADDRESS_TO_USERID_PATH)
members_df = pd.read_csv(COMMUNITY_MEMBERS_PATH)
manifest_df = pd.read_csv(COMMON_MANIFEST_PATH)
finalized_common_df = pd.read_csv(FINALIZED_COMMON_PATH)

print("\n[OK] Input files loaded successfully.")


# =====================================================================================
# 4. Normalize data types
# =====================================================================================

address_df["user_id"] = address_df["user_id"].astype(int)
address_df["address"] = address_df["address"].astype(str)

members_df["community_id"] = members_df["community_id"].astype(int)
members_df["user_id"] = members_df["user_id"].astype(int)

manifest_df["community_id"] = manifest_df["community_id"].astype(int)
manifest_df["finalized_Ak"] = manifest_df["finalized_Ak"].astype(int)

finalized_common_df["community_id"] = finalized_common_df["community_id"].astype(int)

print("[OK] Data types normalized.")


# =====================================================================================
# 5. Validate manifest communities
# =====================================================================================

manifest_communities = sorted(
    manifest_df["community_id"].astype(int).unique().tolist()
)

if manifest_communities != sorted(COMMON_COMMUNITIES):
    raise RuntimeError(
        "Manifest communities do not match COMMON_COMMUNITIES.\n"
        f"Manifest: {manifest_communities}\n"
        f"Expected: {sorted(COMMON_COMMUNITIES)}"
    )

print("\n[OK] Manifest communities match COMMON_COMMUNITIES.")
print("Common communities:", manifest_communities)


# =====================================================================================
# 6. Validate manifest status
# =====================================================================================

if "validation_status" in manifest_df.columns:
    invalid_status = manifest_df[
        manifest_df["validation_status"] != "valid_finalized"
    ]

    if not invalid_status.empty:
        raise RuntimeError(
            "Some manifest communities are not valid_finalized:\n"
            + invalid_status.to_string(index=False)
        )

print("[OK] All manifest communities are valid_finalized.")


if "recommendation_file_exists" in manifest_df.columns:
    if not manifest_df["recommendation_file_exists"].astype(bool).all():
        raise RuntimeError(
            "Some recommendation files are marked as missing in the manifest."
        )

print("[OK] All manifest communities have recommendation files.")


# =====================================================================================
# 7. Build user_id → blockchain address mapping
# =====================================================================================

user_to_address = {
    int(row["user_id"]): Web3.to_checksum_address(row["address"])
    for _, row in address_df.iterrows()
}

print("\nTotal mapped user IDs:", len(user_to_address))


# =====================================================================================
# 8. Build trusted-node table from finalized A_k values
# =====================================================================================

trusted_nodes_df = manifest_df[
    ["community_id", "finalized_Ak"]
].copy()

trusted_nodes_df = trusted_nodes_df.rename(
    columns={
        "finalized_Ak": "trusted_node_user_id"
    }
)

trusted_nodes_df["trusted_node_address"] = trusted_nodes_df["trusted_node_user_id"].map(
    user_to_address
)

if trusted_nodes_df["trusted_node_address"].isna().any():
    missing = trusted_nodes_df[
        trusted_nodes_df["trusted_node_address"].isna()
    ]

    raise RuntimeError(
        "Missing blockchain addresses for some finalized A_k values:\n"
        + missing.to_string(index=False)
    )

trusted_nodes_df["trusted_node_address"] = trusted_nodes_df["trusted_node_address"].apply(
    Web3.to_checksum_address
)

print("\n[OK] Trusted-node addresses mapped from finalized A_k values.")


# =====================================================================================
# 9. Verify that every A_k address is unlocked in Ganache
# =====================================================================================

ganache_accounts_lower = set(
    Web3.to_checksum_address(a).lower()
    for a in accounts
)

missing_unlocked_accounts = []

for _, row in trusted_nodes_df.iterrows():
    cid = int(row["community_id"])
    ak_user_id = int(row["trusted_node_user_id"])
    ak_address = Web3.to_checksum_address(row["trusted_node_address"])

    if ak_address.lower() not in ganache_accounts_lower:
        missing_unlocked_accounts.append(
            {
                "community_id": cid,
                "trusted_node_user_id": ak_user_id,
                "trusted_node_address": ak_address
            }
        )

if missing_unlocked_accounts:
    missing_df = pd.DataFrame(missing_unlocked_accounts)
    raise RuntimeError(
        "Some A_k addresses are not available as unlocked Ganache accounts:\n"
        + missing_df.to_string(index=False)
    )

print("[OK] All A_k addresses are available as unlocked Ganache accounts.")


# =====================================================================================
# 10. Save trusted-node mapping
# =====================================================================================

TRUSTED_NODES_OUTPUT_PATH = (
    OUTPUTS_DIR / "05_lastfm_trusted_nodes_common_from_finalized_announcement.csv"
)

trusted_nodes_df.to_csv(
    TRUSTED_NODES_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nTrusted nodes for common communities:")
print(trusted_nodes_df.to_string(index=False))

print("\n[OK] Trusted-node mapping saved to:")
print(TRUSTED_NODES_OUTPUT_PATH)


# =====================================================================================
# 11. Final summary
# =====================================================================================

print("\n" + "=" * 100)
print("[FINAL SUMMARY]")
print("Loaded contract:", loaded_contract_address)
print("Contract owner:", contract_owner)
print("Common communities count:", len(COMMON_COMMUNITIES))
print("Trusted nodes count:", len(trusted_nodes_df))
print("Total mapped user IDs:", len(user_to_address))
print("=" * 100)

print("\n[OK] Cell 6 completed.")

Cell 6 — Load RecommendationDelivery Contract and Common Last.fm Data
[OK] Loaded RecommendationDelivery contract at:
0x5071DE85Ef0917C7aAf5943Fe98e2806A0C637bd

Contract owner: 0xf24B340783Aa1bbE02A67eA48557B3aD2b7f3e92
Expected owner: 0xf24B340783Aa1bbE02A67eA48557B3aD2b7f3e92
[OK] Contract owner matches OWNER_ACCOUNT.

[OK] Input files loaded successfully.
[OK] Data types normalized.

[OK] Manifest communities match COMMON_COMMUNITIES.
Common communities: [28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506]
[OK] All manifest communities are valid_finalized.
[OK] All manifest communities have recommendation files.

Total mapped user IDs: 783

[OK] Trusted-node addresses mapped from finalized A_k values.
[OK] All A_k addresses are available as unlocked Ganache accounts.

Trusted nodes for common communities:
 community_id  trusted_node_user_id                       trusted_node_address
           28                  1858 0x59239a02DDF3eF95057C9504993B29704e785bfe
           30  

### فائدة Cell 7 ببساطة

هذه الخلية تحدد إعدادات إيصال التوصيات:

DELIVERY_TOP_N = 5
DELIVERY_VERSION = 1

يعني سنأخذ من ملفات Top-20 فقط أول Top-5 توصيات لكل مستخدم، ونخزن Hash لهذه الحزمة داخل العقد.

كذلك تنشئ مجلد خاص بنتائج الإيصال:

RECOMMENDATION_DELIVERY_TOP5_V1

وتقرأ ملفات التوصيات من ملف:

lastfm_common_communities_manifest.csv

حتى يكون الإيصال فقط للمجتمعات المشتركة الـ13.

### Cell 7 — Delivery Configuration

In [7]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 7 — Delivery Configuration
# =====================================================================================
#
# Purpose:
#   هذه الخلية تحدد إعدادات مرحلة إيصال التوصيات.
#
# What this cell does:
#   1. تحدد عدد التوصيات المراد إيصالها لكل مستخدم:
#        DELIVERY_TOP_N = 5
#
#   2. تحدد نسخة الإيصال:
#        DELIVERY_VERSION = 1
#
#   3. تنشئ مجلدًا خاصًا بنتائج هذه التجربة:
#        03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1
#
#   4. تنشئ مجلد Payloads:
#        03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1\payloads
#
#   5. تقرأ ملفات التوصيات من manifest الخاص بالمجتمعات المشتركة.
#
# Why DELIVERY_VERSION is important:
#   إذا تم تنفيذ Cell 10 مرة ثانية على نفس العقد وبنفس:
#       community_id + user_address + version + topN
#   فإن العقد سيرفض التخزين المكرر.
#
#   لذلك إذا أردت إعادة تجربة الإيصال على نفس العقد لاحقًا،
#   غيّر DELIVERY_VERSION إلى 2 أو 3.
#
# =====================================================================================


print("=" * 100)
print("Cell 7 — Delivery Configuration")
print("=" * 100)


# =====================================================================================
# 1. Delivery settings
# =====================================================================================

# عدد التوصيات المراد إيصالها لكل مستخدم.
# نستخدم Top-5 من ملفات Top-20.
DELIVERY_TOP_N = 5

# رقم نسخة الإيصال.
# إذا أعدت تنفيذ التخزين على نفس العقد، غيّرها إلى 2.
DELIVERY_VERSION = 1


# =====================================================================================
# 2. Output directories for this delivery run
# =====================================================================================

DELIVERY_OUTPUT_DIR = (
    OUTPUTS_DIR
    / f"RECOMMENDATION_DELIVERY_TOP{DELIVERY_TOP_N}_V{DELIVERY_VERSION}"
)

PAYLOAD_DIR = DELIVERY_OUTPUT_DIR / "payloads"

DELIVERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)


print("DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("DELIVERY_VERSION:", DELIVERY_VERSION)
print("DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)
print("PAYLOAD_DIR:", PAYLOAD_DIR)


# =====================================================================================
# 3. Build recommendation-file list from the common-community manifest
# =====================================================================================

if "included_in_delivery" not in manifest_df.columns:
    raise RuntimeError(
        "Column included_in_delivery not found in manifest_df."
    )

if "recommendation_file" not in manifest_df.columns:
    raise RuntimeError(
        "Column recommendation_file not found in manifest_df."
    )

RECOMMENDATION_FILES = [
    Path(row["recommendation_file"])
    for _, row in manifest_df.sort_values("community_id").iterrows()
    if bool(row["included_in_delivery"])
]


# =====================================================================================
# 4. Validate recommendation files
# =====================================================================================

print("\nRecommendation files used in delivery:")

missing_files = []

for p in RECOMMENDATION_FILES:
    exists = p.exists()
    print(exists, "=>", p)

    if not exists:
        missing_files.append(p)

if missing_files:
    raise FileNotFoundError(
        "Some recommendation files are missing:\n"
        + "\n".join(str(p) for p in missing_files)
    )


# =====================================================================================
# 5. Validate number of files against common communities
# =====================================================================================

if len(RECOMMENDATION_FILES) != len(COMMON_COMMUNITIES):
    raise RuntimeError(
        f"Expected {len(COMMON_COMMUNITIES)} recommendation files, "
        f"but got {len(RECOMMENDATION_FILES)}"
    )

print("\n[FINAL CHECK]")
print("All common-community recommendation files are ready for delivery.")
print("Number of recommendation files:", len(RECOMMENDATION_FILES))
print("Number of common communities:", len(COMMON_COMMUNITIES))

print("\n[OK] Cell 7 completed.")

Cell 7 — Delivery Configuration
DELIVERY_TOP_N: 5
DELIVERY_VERSION: 1
DELIVERY_OUTPUT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1
PAYLOAD_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1\payloads

Recommendation files used in delivery:
True => D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_28\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28.csv
True => D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_30\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM30\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_PO

### Cell 7B — Corrected Delivery Version

In [16]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 7B — Corrected Delivery Version
# =====================================================================================
#
# Purpose:
#   استخدام نسخة جديدة للتخزين المصحح حتى لا تختلط مع نتائج DELIVERY_VERSION = 1.
# =====================================================================================

print("=" * 100)
print("Cell 7B — Corrected Delivery Version")
print("=" * 100)

DELIVERY_TOP_N = 5
DELIVERY_VERSION = 2

DELIVERY_OUTPUT_DIR = (
    OUTPUTS_DIR
    / f"RECOMMENDATION_DELIVERY_TOP{DELIVERY_TOP_N}_V{DELIVERY_VERSION}"
)

PAYLOAD_DIR = DELIVERY_OUTPUT_DIR / "payloads"

DELIVERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("DELIVERY_VERSION:", DELIVERY_VERSION)
print("DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)
print("PAYLOAD_DIR:", PAYLOAD_DIR)

print("\n[OK] Cell 7B completed.")

Cell 7B — Corrected Delivery Version
DELIVERY_TOP_N: 5
DELIVERY_VERSION: 2
DELIVERY_OUTPUT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2
PAYLOAD_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2\payloads

[OK] Cell 7B completed.


### فائدة Cell 8 ببساطة

هذه الخلية لا تنفذ الإيصال ولا ترسل معاملات مباشرة، لكنها تجهز كل الدوال التي ستحتاجها الخلايا اللاحقة.

بمعنى آخر، هي “صندوق الأدوات” الخاص بمرحلة الإيصال.

وظيفتها الأساسية:

1. استخراج رقم المجتمع من اسم ملف التوصيات.
2. جلب عنوان A_k لكل مجتمع.
3. تحويل user_id إلى عنوان Blockchain.
4. بناء Payload التوصيات.
5. حساب Hash ثابت للتوصيات.
6. تسجيل A_k داخل العقد.
7. تسجيل أعضاء المجتمع داخل العقد.
8. تخزين Hash التوصيات من حساب A_k الصحيح.
9. توحيد أسماء أعمدة ملفات التوصيات.
10. استخراج Top-5 من ملفات Top-20.
    ### Cell 8 — Helper Functions for Last.fm Recommendation Delivery

In [8]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 8 — Helper Functions for Last.fm Recommendation Delivery
# =====================================================================================
#
# Purpose:
#   هذه الخلية تجمع الدوال المساعدة المطلوبة لمرحلة إيصال التوصيات.
#
# Important:
#   هذه الخلية لا تخزن Hash ولا ترسل توصيات.
#   هي فقط تجهز الأدوات التي ستستخدمها الخلايا التالية.
#
# Main functions:
#   1. extract_community_id_from_path
#      استخراج community_id من اسم ملف التوصيات.
#
#   2. get_trusted_node_address_for_community
#      جلب عنوان A_k النهائي للمجتمع.
#
#   3. user_address_for_dataset_user
#      تحويل user_id إلى عنوان blockchain.
#
#   4. compute_recommendation_hash
#      حساب Hash لحزمة التوصيات.
#
#   5. ensure_authorized_node
#      تسجيل A_k داخل عقد RecommendationDelivery.
#
#   6. register_community_members_onchain
#      تسجيل أعضاء المجتمع داخل العقد.
#
#   7. store_hash_onchain
#      تخزين Hash التوصيات من حساب A_k الصحيح.
#
#   8. normalize_recommendation_columns
#      توحيد أسماء الأعمدة داخل ملفات التوصيات.
#
#   9. build_user_topn_recommendations
#      استخراج Top-N recommendation list لكل مستخدم.
#
# =====================================================================================


def extract_community_id_from_path(path: Path) -> int:
    """
    Extract community_id from the recommendation file name.

    Supported patterns:
        comm_28
        COMM_28
        COMM28
        community_28
        Community_28
    """
    file_name = path.name

    patterns = [
        r"comm_(\d+)",
        r"COMM_(\d+)",
        r"COMM(\d+)",
        r"community_(\d+)",
        r"Community_(\d+)",
    ]

    for pat in patterns:
        match = re.search(pat, file_name)
        if match:
            return int(match.group(1))

    raise ValueError(
        f"Cannot extract community_id from file name: {file_name}"
    )


def get_trusted_node_address_for_community(community_id: int) -> str:
    """
    Return the blockchain address of the finalized A_k for a given community.
    """
    row = trusted_nodes_df[
        trusted_nodes_df["community_id"] == int(community_id)
    ]

    if row.empty:
        raise ValueError(
            f"No trusted node found for community_id={community_id}"
        )

    return Web3.to_checksum_address(
        row.iloc[0]["trusted_node_address"]
    )


def get_trusted_node_user_id_for_community(community_id: int) -> int:
    """
    Return the user_id of the finalized A_k for a given community.
    """
    row = trusted_nodes_df[
        trusted_nodes_df["community_id"] == int(community_id)
    ]

    if row.empty:
        raise ValueError(
            f"No trusted node found for community_id={community_id}"
        )

    return int(row.iloc[0]["trusted_node_user_id"])


def user_address_for_dataset_user(user_id: int) -> str:
    """
    Convert Last.fm user_id into its blockchain address.

    The address must already exist in address_to_userid.csv
    generated during the announcement stage.
    """
    user_id = int(user_id)

    if user_id not in user_to_address:
        raise ValueError(
            f"user_id={user_id} does not exist in Last.fm address_to_userid.csv"
        )

    return Web3.to_checksum_address(
        user_to_address[user_id]
    )


def build_recommendation_payload_for_hash(
    user_address: str,
    community_id: int,
    version: int,
    top_n: int,
    recommendations: list,
    nonce: str
) -> dict:
    """
    Build the recommendation payload used for hash computation.

    The full payload remains off-chain.
    Only its hash is stored on-chain.
    """
    return {
        "dataset": "Last.fm",
        "user_address": Web3.to_checksum_address(user_address),
        "community_id": int(community_id),
        "version": int(version),
        "top_n": int(top_n),
        "recommendations": recommendations,
        "nonce": str(nonce)
    }


def canonical_json(data: dict) -> str:
    """
    Convert data into deterministic JSON.

    This is important because the same payload must always produce
    the same hash.
    """
    return json.dumps(
        data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )


def compute_recommendation_hash(
    user_address: str,
    community_id: int,
    version: int,
    top_n: int,
    recommendations: list,
    nonce: str
):
    """
    Compute keccak256 hash for the recommendation payload.
    """
    data = build_recommendation_payload_for_hash(
        user_address=user_address,
        community_id=community_id,
        version=version,
        top_n=top_n,
        recommendations=recommendations,
        nonce=nonce
    )

    message = canonical_json(data)

    return w3.keccak(text=message)


def ensure_authorized_node(community_id: int):
    """
    Register finalized A_k for the community inside RecommendationDelivery.

    If A_k is already registered, the function does not send a new transaction.
    """
    trusted_node_address = get_trusted_node_address_for_community(
        community_id
    )

    current_node = contract.functions.authorizedNode(
        int(community_id)
    ).call()

    if current_node.lower() == trusted_node_address.lower():
        print(
            f"[A_k] Already set for community {community_id}: "
            f"{trusted_node_address}"
        )
        return trusted_node_address, None

    tx_hash = contract.functions.setAuthorizedNode(
        int(community_id),
        trusted_node_address
    ).transact(
        {
            "from": OWNER_ACCOUNT,
            "gas": 500_000
        }
    )

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError(
            f"Failed to set A_k for community {community_id}"
        )

    print(
        f"[A_k] Set for community {community_id}: "
        f"{trusted_node_address}"
    )

    return trusted_node_address, tx_hash.hex()


def register_community_members_onchain(
    community_id: int,
    member_addresses: list,
    batch_size: int = 100
):
    """
    Register community members inside RecommendationDelivery.

    This prevents A_k from storing recommendation hashes for users
    outside its community.
    """
    member_addresses = [
        Web3.to_checksum_address(x)
        for x in member_addresses
    ]

    unique_members = sorted(set(member_addresses))

    print(
        f"[MEMBERS] Registering {len(unique_members)} members "
        f"for community {community_id}"
    )

    tx_hashes = []

    current_count_before = int(
        contract.functions.communityMemberCount(
            int(community_id)
        ).call()
    )

    for start in range(0, len(unique_members), batch_size):
        batch = unique_members[start:start + batch_size]

        tx_hash = contract.functions.registerCommunityMembers(
            int(community_id),
            batch
        ).transact(
            {
                "from": OWNER_ACCOUNT,
                "gas": 6_000_000
            }
        )

        receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

        if receipt.status != 1:
            raise RuntimeError(
                f"Failed to register members for community={community_id}, "
                f"batch_start={start}"
            )

        tx_hashes.append(tx_hash.hex())

        print(
            f"[MEMBERS] community={community_id}, "
            f"batch={start // batch_size + 1}, "
            f"tx={tx_hash.hex()}"
        )

    current_count_after = int(
        contract.functions.communityMemberCount(
            int(community_id)
        ).call()
    )

    print(
        f"[MEMBERS] Before={current_count_before}, "
        f"After={current_count_after}"
    )

    return tx_hashes


def store_hash_onchain(
    community_id: int,
    user_address: str,
    version: int,
    top_n: int,
    recommendation_hash
):
    """
    Store recommendation hash on-chain.

    The transaction is sent from the finalized A_k address of that community.
    """
    trusted_node_address = get_trusted_node_address_for_community(
        community_id
    )

    exists = contract.functions.recordExists(
        int(community_id),
        Web3.to_checksum_address(user_address),
        int(version),
        int(top_n)
    ).call()

    if exists:
        return None, trusted_node_address

    tx_hash = contract.functions.storeRecommendationHash(
        int(community_id),
        Web3.to_checksum_address(user_address),
        int(version),
        int(top_n),
        recommendation_hash
    ).transact(
        {
            "from": trusted_node_address,
            "gas": 500_000
        }
    )

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError(
            f"Failed to store hash for community={community_id}, "
            f"user={user_address}, version={version}, topN={top_n}"
        )

    return tx_hash.hex(), trusted_node_address


def normalize_recommendation_columns(recs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize recommendation file columns.

    Supported user columns:
        user, user_id, userid, uid

    Supported item columns:
        item, item_id, artist, artist_id, artistid,
        recommended_item, recommended_artist, rec_item

    Supported rank columns:
        rank, rec_rank, position
    """
    cols = {
        c.lower().strip(): c
        for c in recs_df.columns
    }

    user_candidates = [
        "user", "user_id", "userid", "uid"
    ]

    item_candidates = [
        "item", "item_id", "artist", "artist_id", "artistid",
        "recommended_item", "recommended_artist", "rec_item"
    ]

    rank_candidates = [
        "rank", "rec_rank", "position"
    ]

    user_col = next(
        (cols[c] for c in user_candidates if c in cols),
        None
    )

    item_col = next(
        (cols[c] for c in item_candidates if c in cols),
        None
    )

    rank_col = next(
        (cols[c] for c in rank_candidates if c in cols),
        None
    )

    if user_col is None:
        raise ValueError(
            f"Cannot find user column. Available columns: {list(recs_df.columns)}"
        )

    if item_col is None:
        raise ValueError(
            f"Cannot find item/artist column. Available columns: {list(recs_df.columns)}"
        )

    if rank_col is None:
        raise ValueError(
            f"Cannot find rank column. Available columns: {list(recs_df.columns)}"
        )

    out = recs_df.copy()

    out = out.rename(
        columns={
            user_col: "user",
            item_col: "item",
            rank_col: "rank"
        }
    )

    out["user"] = out["user"].astype(int)
    out["item"] = out["item"].astype(int)
    out["rank"] = out["rank"].astype(int)

    return out


def build_user_topn_recommendations(
    recs_df: pd.DataFrame,
    user_id: int,
    top_n: int
) -> list:
    """
    Build the Top-N recommendation list for one user.

    Only rank and item are included in the payload.
    Prediction scores are not included to avoid floating-point
    representation issues in hashing.
    """
    user_recs = recs_df[
        recs_df["user"].astype(int) == int(user_id)
    ].copy()

    if user_recs.empty:
        return []

    user_recs = user_recs.sort_values("rank")
    user_recs = user_recs[user_recs["rank"] <= top_n]

    recommendations = []

    for _, row in user_recs.iterrows():
        recommendations.append(
            {
                "rank": int(row["rank"]),
                "item": int(row["item"])
            }
        )

    return recommendations


print("=" * 100)
print("Cell 8 — Helper Functions for Last.fm Recommendation Delivery")
print("=" * 100)

print("[OK] Helper functions were defined successfully.")

print("\nAvailable helper functions:")
print("1. extract_community_id_from_path")
print("2. get_trusted_node_address_for_community")
print("3. get_trusted_node_user_id_for_community")
print("4. user_address_for_dataset_user")
print("5. compute_recommendation_hash")
print("6. ensure_authorized_node")
print("7. register_community_members_onchain")
print("8. store_hash_onchain")
print("9. normalize_recommendation_columns")
print("10. build_user_topn_recommendations")

print("\n[OK] Cell 8 completed.")

Cell 8 — Helper Functions for Last.fm Recommendation Delivery
[OK] Helper functions were defined successfully.

Available helper functions:
1. extract_community_id_from_path
2. get_trusted_node_address_for_community
3. get_trusted_node_user_id_for_community
4. user_address_for_dataset_user
5. compute_recommendation_hash
6. ensure_authorized_node
7. register_community_members_onchain
8. store_hash_onchain
9. normalize_recommendation_columns
10. build_user_topn_recommendations

[OK] Cell 8 completed.


### Cell 8A-v2 — Re-Diagnose User-ID Alignment After Fix
لاحظ هذا
8A-v2
يجب ان يكون بعد ال 8

In [14]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 8A-v2 — Fix Last.fm Recommendation Column Mapping without Duplicate Columns
# =====================================================================================
#
# Purpose:
#   تصحيح قراءة ملفات توصيات Last.fm بدون إنشاء أعمدة مكررة.
#
# Problem:
#   الملفات تحتوي عمود user وعمود userID.
#   إذا عملنا rename من userID إلى user مع بقاء user الأصلي،
#   يصبح لدينا عمودان باسم user، وهذا يسبب الخطأ:
#       AttributeError: 'DataFrame' object has no attribute 'unique'
#
# Correct mapping:
#   userID   -> user
#   artistID -> item
#   rank     -> rank
# =====================================================================================

print("=" * 100)
print("Cell 8A-v2 — Fix Last.fm Recommendation Column Mapping without Duplicate Columns")
print("=" * 100)


def normalize_recommendation_columns(recs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize Last.fm recommendation file columns using original IDs.

    This version avoids duplicate column names by constructing a new DataFrame.
    """

    cols = {
        c.lower().strip(): c
        for c in recs_df.columns
    }

    # For these Last.fm files, userID is the original user identifier.
    if "userid" in cols:
        user_col = cols["userid"]
    elif "user_id" in cols:
        user_col = cols["user_id"]
    else:
        raise ValueError(
            "Cannot find original userID column. "
            f"Available columns: {list(recs_df.columns)}"
        )

    # For these Last.fm files, artistID is the original item/artist identifier.
    if "artistid" in cols:
        item_col = cols["artistid"]
    elif "artist_id" in cols:
        item_col = cols["artist_id"]
    elif "item_id" in cols:
        item_col = cols["item_id"]
    else:
        raise ValueError(
            "Cannot find original artistID/item column. "
            f"Available columns: {list(recs_df.columns)}"
        )

    if "rank" in cols:
        rank_col = cols["rank"]
    elif "rec_rank" in cols:
        rank_col = cols["rec_rank"]
    elif "position" in cols:
        rank_col = cols["position"]
    else:
        raise ValueError(
            "Cannot find rank column. "
            f"Available columns: {list(recs_df.columns)}"
        )

    out = pd.DataFrame({
        "user": recs_df[user_col].astype(int),
        "item": recs_df[item_col].astype(int),
        "rank": recs_df[rank_col].astype(int)
    })

    # Optional: keep test flag if available
    if "is_positive_in_test" in recs_df.columns:
        out["is_positive_in_test"] = recs_df["is_positive_in_test"]

    print("[COLUMN MAPPING]")
    print("Selected user column:", user_col, "=> user")
    print("Selected item column:", item_col, "=> item")
    print("Selected rank column:", rank_col, "=> rank")
    print("Output columns:", list(out.columns))

    return out


def build_user_topn_recommendations(
    recs_df: pd.DataFrame,
    user_id: int,
    top_n: int
) -> list:
    """
    Build Top-N recommendation list for one original Last.fm userID.
    """

    user_recs = recs_df[
        recs_df["user"].astype(int) == int(user_id)
    ].copy()

    if user_recs.empty:
        return []

    user_recs = user_recs.sort_values("rank")
    user_recs = user_recs[user_recs["rank"] <= int(top_n)]

    recommendations = []

    for _, row in user_recs.iterrows():
        recommendations.append(
            {
                "rank": int(row["rank"]),
                "item": int(row["item"])
            }
        )

    return recommendations


print("[OK] normalize_recommendation_columns was corrected without duplicate columns.")
print("[OK] build_user_topn_recommendations was corrected.")
print("[OK] Cell 8A-v2 completed.")

Cell 8A-v2 — Fix Last.fm Recommendation Column Mapping without Duplicate Columns
[OK] normalize_recommendation_columns was corrected without duplicate columns.
[OK] build_user_topn_recommendations was corrected.
[OK] Cell 8A-v2 completed.


### فائدة Cell 9 ببساطة

هذه أول خلية تنفذ معاملات فعلية على عقد RecommendationDelivery.

وظيفتها:

1. تسجيل A_k النهائي لكل مجتمع داخل عقد الإيصال.
2. تسجيل أعضاء كل مجتمع داخل العقد.
3. التحقق أن عدد الأعضاء المسجلين يطابق العدد المتوقع.
4. حفظ تقرير بالمعاملات والنتائج.

هذه الخطوة مهمة لأن العقد لن يسمح بتخزين Hash التوصيات لاحقًا إلا إذا تحقق شرطان:

1. المرسل هو A_k المعتمد لذلك المجتمع.
2. المستخدم الهدف عضو مسجل داخل ذلك المجتمع.
### Cell 9 — Register A_k and Community Members on RecommendationDelivery Contract

In [9]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 9 — Register A_k and Community Members on RecommendationDelivery Contract
# =====================================================================================
#
# Purpose:
#   هذه الخلية تسجل القادة النهائيين A_k وأعضاء المجتمعات المشتركة داخل عقد
#   RecommendationDelivery.
#
# What this cell does:
#   1. لكل مجتمع مشترك:
#        - تسجل A_k النهائي باستخدام setAuthorizedNode.
#
#   2. تسجل أعضاء المجتمع داخل العقد باستخدام registerCommunityMembers.
#
#   3. تتحقق أن عدد الأعضاء على العقد يطابق عدد الأعضاء المتوقع.
#
#   4. تحفظ تقرير التنفيذ في:
#        03_outputs\RECOMMENDATION_DELIVERY_TOP5_V1
#
# Important:
#   هذه الخلية يجب تشغيلها مرة واحدة على العقد الحالي.
#   إذا أعدت تشغيلها، الكود يحاول تجنب إعادة تسجيل الأعضاء المسجلين مسبقًا
#   إذا كانت دالة isCommunityMember متاحة في العقد.
#
# =====================================================================================


print("=" * 100)
print("Cell 9 — Register A_k and Community Members on RecommendationDelivery Contract")
print("=" * 100)


# =====================================================================================
# 1. Output report paths
# =====================================================================================

REGISTER_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "06_register_authorized_nodes_and_members_report.csv"
)

REGISTER_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "06_register_authorized_nodes_and_members_summary.json"
)


# =====================================================================================
# 2. Helper: check whether a function exists in ABI
# =====================================================================================

def abi_has_function(abi, function_name: str) -> bool:
    return any(
        item.get("type") == "function" and item.get("name") == function_name
        for item in abi
    )


HAS_IS_COMMUNITY_MEMBER = abi_has_function(loaded_abi, "isCommunityMember")

print("Contract has isCommunityMember getter:", HAS_IS_COMMUNITY_MEMBER)


# =====================================================================================
# 3. Helper: filter already-registered members
# =====================================================================================

def filter_unregistered_members(community_id: int, member_addresses: list) -> list:
    """
    Return only addresses that are not yet registered on-chain.

    If isCommunityMember is not available, the function returns the full list.
    """
    unique_members = sorted(
        set(Web3.to_checksum_address(addr) for addr in member_addresses)
    )

    if not HAS_IS_COMMUNITY_MEMBER:
        return unique_members

    pending_members = []

    for addr in unique_members:
        already_registered = contract.functions.isCommunityMember(
            int(community_id),
            Web3.to_checksum_address(addr)
        ).call()

        if not already_registered:
            pending_members.append(addr)

    return pending_members


# =====================================================================================
# 4. Register A_k and community members
# =====================================================================================

registration_rows = []

for community_id in sorted(COMMON_COMMUNITIES):

    print("\n" + "-" * 100)
    print(f"Processing community {community_id}")
    print("-" * 100)

    # -------------------------------------------------------------------------
    # 4.1 Get finalized A_k
    # -------------------------------------------------------------------------

    trusted_node_user_id = get_trusted_node_user_id_for_community(community_id)
    trusted_node_address = get_trusted_node_address_for_community(community_id)

    print("Trusted node user_id:", trusted_node_user_id)
    print("Trusted node address:", trusted_node_address)

    # -------------------------------------------------------------------------
    # 4.2 Register A_k in the RecommendationDelivery contract
    # -------------------------------------------------------------------------

    trusted_node_address_onchain, ak_tx_hash = ensure_authorized_node(
        community_id
    )

    authorized_node_after = Web3.to_checksum_address(
        contract.functions.authorizedNode(int(community_id)).call()
    )

    ak_matches = (
        authorized_node_after.lower()
        == Web3.to_checksum_address(trusted_node_address).lower()
    )

    if not ak_matches:
        raise RuntimeError(
            f"A_k mismatch after registration for community={community_id}"
        )

    print("[OK] A_k verified on-chain.")

    # -------------------------------------------------------------------------
    # 4.3 Prepare community members
    # -------------------------------------------------------------------------

    community_members = members_df[
        members_df["community_id"].astype(int) == int(community_id)
    ].copy()

    if community_members.empty:
        raise RuntimeError(
            f"No members found in members_df for community={community_id}"
        )

    expected_member_user_ids = sorted(
        community_members["user_id"].astype(int).unique().tolist()
    )

    expected_member_addresses = []

    missing_member_addresses = []

    for uid in expected_member_user_ids:
        try:
            expected_member_addresses.append(
                user_address_for_dataset_user(uid)
            )
        except Exception:
            missing_member_addresses.append(uid)

    if missing_member_addresses:
        raise RuntimeError(
            f"Missing blockchain addresses for members in community={community_id}: "
            f"{missing_member_addresses}"
        )

    expected_member_addresses = sorted(
        set(Web3.to_checksum_address(addr) for addr in expected_member_addresses)
    )

    expected_member_count = len(expected_member_addresses)

    print("Expected member count:", expected_member_count)

    # -------------------------------------------------------------------------
    # 4.4 Avoid duplicate registration if possible
    # -------------------------------------------------------------------------

    current_count_before = int(
        contract.functions.communityMemberCount(int(community_id)).call()
    )

    pending_member_addresses = filter_unregistered_members(
        community_id,
        expected_member_addresses
    )

    pending_member_count = len(pending_member_addresses)

    print("On-chain member count before:", current_count_before)
    print("Pending members to register:", pending_member_count)

    # -------------------------------------------------------------------------
    # 4.5 Register pending members
    # -------------------------------------------------------------------------

    if pending_member_count > 0:
        member_tx_hashes = register_community_members_onchain(
            community_id=community_id,
            member_addresses=pending_member_addresses,
            batch_size=100
        )
    else:
        member_tx_hashes = []
        print("[MEMBERS] No pending members. Registration skipped.")

    current_count_after = int(
        contract.functions.communityMemberCount(int(community_id)).call()
    )

    print("On-chain member count after:", current_count_after)

    # -------------------------------------------------------------------------
    # 4.6 Final verification for this community
    # -------------------------------------------------------------------------

    # If the contract counts unique members, the final count should match expected_member_count.
    if current_count_after != expected_member_count:
        raise RuntimeError(
            f"Member count mismatch for community={community_id}. "
            f"Expected={expected_member_count}, On-chain={current_count_after}"
        )

    print("[OK] Member count verified on-chain.")

    # -------------------------------------------------------------------------
    # 4.7 Save row
    # -------------------------------------------------------------------------

    registration_rows.append(
        {
            "community_id": int(community_id),
            "trusted_node_user_id": int(trusted_node_user_id),
            "trusted_node_address": Web3.to_checksum_address(trusted_node_address),
            "authorized_node_onchain": authorized_node_after,
            "ak_matches": bool(ak_matches),
            "ak_registration_tx": ak_tx_hash if ak_tx_hash is not None else "",
            "expected_member_count": int(expected_member_count),
            "onchain_member_count_before": int(current_count_before),
            "pending_member_count_registered": int(pending_member_count),
            "onchain_member_count_after": int(current_count_after),
            "member_count_matches": bool(current_count_after == expected_member_count),
            "member_registration_tx_count": int(len(member_tx_hashes)),
            "member_registration_txs": "|".join(member_tx_hashes),
            "status": "registered_and_verified"
        }
    )


# =====================================================================================
# 5. Save registration report
# =====================================================================================

registration_report_df = pd.DataFrame(registration_rows)

registration_report_df.to_csv(
    REGISTER_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "contract_name": "RecommendationDelivery",
    "contract_address": loaded_contract_address,
    "chain_id": int(w3.eth.chain_id),
    "owner_account": OWNER_ACCOUNT,
    "delivery_top_n": int(DELIVERY_TOP_N),
    "delivery_version": int(DELIVERY_VERSION),
    "common_communities": [int(x) for x in sorted(COMMON_COMMUNITIES)],
    "common_communities_count": int(len(COMMON_COMMUNITIES)),
    "registered_communities_count": int(len(registration_report_df)),
    "all_ak_match": bool(registration_report_df["ak_matches"].all()),
    "all_member_counts_match": bool(registration_report_df["member_count_matches"].all()),
    "total_member_registration_transactions": int(
        registration_report_df["member_registration_tx_count"].sum()
    ),
    "report_path": str(REGISTER_REPORT_PATH),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

REGISTER_SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# =====================================================================================
# 6. Final report
# =====================================================================================

print("\n" + "=" * 100)
print("[REGISTRATION REPORT]")
print(registration_report_df.to_string(index=False))

print("\n[OK] Registration report saved to:")
print(REGISTER_REPORT_PATH)

print("\n[OK] Registration summary saved to:")
print(REGISTER_SUMMARY_PATH)

print("\n" + "=" * 100)
print("[FINAL CHECK]")
print("All A_k values were registered and verified:", summary["all_ak_match"])
print("All community member counts were verified:", summary["all_member_counts_match"])
print("Registered communities count:", summary["registered_communities_count"])
print("Total member registration transactions:", summary["total_member_registration_transactions"])
print("=" * 100)

print("\n[OK] Cell 9 completed.")

Cell 9 — Register A_k and Community Members on RecommendationDelivery Contract
Contract has isCommunityMember getter: True

----------------------------------------------------------------------------------------------------
Processing community 28
----------------------------------------------------------------------------------------------------
Trusted node user_id: 1858
Trusted node address: 0x59239a02DDF3eF95057C9504993B29704e785bfe
[A_k] Set for community 28: 0x59239a02DDF3eF95057C9504993B29704e785bfe
[OK] A_k verified on-chain.
Expected member count: 38
On-chain member count before: 0
Pending members to register: 38
[MEMBERS] Registering 38 members for community 28
[MEMBERS] community=28, batch=1, tx=3a833cff28b919276b578f990fdacb6bae1a63afca9a9555dd29a49de5e18446
[MEMBERS] Before=0, After=38
On-chain member count after: 38
[OK] Member count verified on-chain.

----------------------------------------------------------------------------------------------------
Processing communi

### Cell 10B

In [15]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 10B — Re-Diagnose User-ID Alignment After Fix
# =====================================================================================
#
# Purpose:
#   إعادة تشخيص تطابق user_id بعد تصحيح قراءة ملفات التوصية في Cell 8A-v2.
#
# Difference from Cell 10A:
#   Cell 10A كان قبل التصحيح، واكتشف أن column "user" هو local index.
#   Cell 10B يعمل بعد التصحيح:
#       userID   -> user
#       artistID -> item
#
# Important:
#   هذه الخلية لا تغيّر العقد ولا ترسل معاملات.
#   فقط تقرأ ملفات التوصية وتقارن المستخدمين مع community_members.
# =====================================================================================

print("=" * 100)
print("Cell 10B — Re-Diagnose User-ID Alignment After Fix")
print("=" * 100)

diagnostic_fixed_rows = []

for rec_file in RECOMMENDATION_FILES:

    community_id = extract_community_id_from_path(rec_file)

    print("\n" + "-" * 100)
    print(f"Re-diagnosing community {community_id}")
    print("-" * 100)

    # Load raw recommendation file
    recs_raw_df = pd.read_csv(rec_file)

    print("Raw columns:", list(recs_raw_df.columns))

    # This now uses the corrected function from Cell 8A-v2
    recs_df = normalize_recommendation_columns(recs_raw_df)

    rec_user_ids = set(
        recs_df["user"].astype(int).unique().tolist()
    )

    community_user_ids = set(
        members_df[
            members_df["community_id"].astype(int) == int(community_id)
        ]["user_id"].astype(int).unique().tolist()
    )

    intersection_users = sorted(
        rec_user_ids.intersection(community_user_ids)
    )

    missing_from_recs = sorted(
        community_user_ids - rec_user_ids
    )

    extra_in_recs = sorted(
        rec_user_ids - community_user_ids
    )

    rec_users_count = len(rec_user_ids)
    community_members_count = len(community_user_ids)
    intersection_count = len(intersection_users)

    coverage_ratio = (
        intersection_count / community_members_count
        if community_members_count > 0
        else 0
    )

    rec_min = min(rec_user_ids) if rec_user_ids else None
    rec_max = max(rec_user_ids) if rec_user_ids else None

    comm_min = min(community_user_ids) if community_user_ids else None
    comm_max = max(community_user_ids) if community_user_ids else None

    print("Recommendation users count:", rec_users_count)
    print("Community members count:", community_members_count)
    print("Intersection count:", intersection_count)
    print("Coverage ratio:", round(coverage_ratio, 4))
    print("Recommendation userID range:", rec_min, "to", rec_max)
    print("Community member user_id range:", comm_min, "to", comm_max)
    print("Sample intersection users:", intersection_users[:10])
    print("Sample missing community users from recs:", missing_from_recs[:10])
    print("Sample extra rec users not in community:", extra_in_recs[:10])

    diagnostic_fixed_rows.append(
        {
            "community_id": int(community_id),
            "recommendation_users_count": int(rec_users_count),
            "community_members_count": int(community_members_count),
            "intersection_count": int(intersection_count),
            "coverage_ratio": float(coverage_ratio),
            "rec_user_min": rec_min,
            "rec_user_max": rec_max,
            "community_user_min": comm_min,
            "community_user_max": comm_max,
            "missing_from_recs_count": int(len(missing_from_recs)),
            "extra_in_recs_count": int(len(extra_in_recs)),
            "sample_intersection_users": str(intersection_users[:20]),
            "sample_missing_from_recs": str(missing_from_recs[:20]),
            "sample_extra_in_recs": str(extra_in_recs[:20]),
            "recommendation_file": str(rec_file)
        }
    )


diagnostic_fixed_df = pd.DataFrame(diagnostic_fixed_rows)

DIAGNOSTIC_FIXED_OUTPUT_PATH = (
    DELIVERY_OUTPUT_DIR / "08B_user_id_alignment_diagnostic_after_fix.csv"
)

diagnostic_fixed_df.to_csv(
    DIAGNOSTIC_FIXED_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 100)
print("[USER-ID ALIGNMENT DIAGNOSTIC AFTER FIX]")
print(diagnostic_fixed_df.to_string(index=False))

print("\n[OK] Diagnostic after fix saved to:")
print(DIAGNOSTIC_FIXED_OUTPUT_PATH)

print("\n[FINAL CHECK]")
print("Average coverage ratio:", diagnostic_fixed_df["coverage_ratio"].mean())
print("Minimum coverage ratio:", diagnostic_fixed_df["coverage_ratio"].min())
print("Maximum coverage ratio:", diagnostic_fixed_df["coverage_ratio"].max())

print(
    "Communities with zero intersection:",
    diagnostic_fixed_df[
        diagnostic_fixed_df["intersection_count"] == 0
    ]["community_id"].tolist()
)

print(
    "Communities with full coverage:",
    diagnostic_fixed_df[
        diagnostic_fixed_df["coverage_ratio"] == 1.0
    ]["community_id"].tolist()
)

print("\n[OK] Cell 10B completed.")

Cell 10B — Re-Diagnose User-ID Alignment After Fix

----------------------------------------------------------------------------------------------------
Re-diagnosing community 28
----------------------------------------------------------------------------------------------------
Raw columns: ['user', 'rank', 'i_idx', 'is_positive_in_test', 'userID', 'artistID']
[COLUMN MAPPING]
Selected user column: userID => user
Selected item column: artistID => item
Selected rank column: rank => rank
Output columns: ['user', 'item', 'rank', 'is_positive_in_test']
Recommendation users count: 38
Community members count: 38
Intersection count: 38
Coverage ratio: 1.0
Recommendation userID range: 2 to 1900
Community member user_id range: 2 to 1900
Sample intersection users: [2, 63, 66, 96, 117, 128, 141, 176, 196, 224]
Sample missing community users from recs: []
Sample extra rec users not in community: []

-------------------------------------------------------------------------------------------------

### Cell 10- ببساطة

هذه الخلية تنفذ جوهر مرحلة إيصال التوصيات:

1. تقرأ ملفات Top-20 للمجتمعات المشتركة.
2. تستخرج Top-5 لكل مستخدم.
3. تبني Payload خارج السلسلة Off-chain.
4. تحفظ Payload كملف JSON محليًا.
5. تحسب Hash لكل Payload.
6. تخزن الـ Hash فقط داخل عقد RecommendationDelivery.
7. ترسل كل معاملة من عنوان A_k الخاص بالمجتمع.

المهم علميًا: التوصيات الكاملة لا تُخزن على البلوكشين، وإنما يتم تخزين Hash فقط للتحقق لاحقًا من سلامة الحزمة.

### Cell 10 — Build Payloads and Store Recommendation Hashes On-Chain


In [17]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 10 — Build Payloads and Store Recommendation Hashes On-Chain
# =====================================================================================
#
# Purpose:
#   هذه الخلية تبني حزم التوصيات Off-chain لكل مستخدم داخل المجتمعات المشتركة،
#   ثم تخزن Hash فقط داخل عقد RecommendationDelivery.
#
# Important:
#   هذه النسخة يجب تشغيلها بعد:
#       Cell 8A-v2  -> corrected userID / artistID mapping
#       Cell 7B     -> DELIVERY_VERSION = 2
#
# What this cell does:
#   1. تقرأ ملفات Top-20 الخاصة بالمجتمعات المشتركة.
#   2. تستخرج Top-N حسب DELIVERY_TOP_N.
#   3. تبني Payload لكل مستخدم.
#   4. تحفظ Payload محليًا كملف JSON.
#   5. تحسب keccak256 hash للحزمة.
#   6. تخزن hash داخل العقد بواسطة A_k الخاص بنفس المجتمع.
#   7. تحفظ تقريرًا مفصلًا بنتائج التخزين.
#
# =====================================================================================


print("=" * 100)
print("Cell 10 — Build Payloads and Store Recommendation Hashes On-Chain")
print("=" * 100)

print("Current DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("Current DELIVERY_VERSION:", DELIVERY_VERSION)
print("Current DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)
print("Current PAYLOAD_DIR:", PAYLOAD_DIR)

if int(DELIVERY_VERSION) != 2:
    raise RuntimeError(
        "DELIVERY_VERSION is not 2. شغّل Cell 7B أولًا قبل إعادة Cell 10."
    )


# =====================================================================================
# 1. Output paths
# =====================================================================================

HASH_STORAGE_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_report.csv"
)

HASH_STORAGE_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_summary.json"
)

PAYLOAD_INDEX_PATH = (
    DELIVERY_OUTPUT_DIR / "07_payload_index.csv"
)


# =====================================================================================
# 2. Optional ABI checks
# =====================================================================================

def abi_has_function_local(abi, function_name: str) -> bool:
    return any(
        item.get("type") == "function" and item.get("name") == function_name
        for item in abi
    )


HAS_GET_RECOMMENDATION_HASH = abi_has_function_local(
    loaded_abi,
    "getRecommendationHash"
)

print("Contract has getRecommendationHash:", HAS_GET_RECOMMENDATION_HASH)


# =====================================================================================
# 3. Helper: convert bytes32/hash to hex string
# =====================================================================================

def hash_to_hex(h) -> str:
    """
    Convert hash value to 0x-prefixed hex string.
    """
    if isinstance(h, str):
        return h if h.startswith("0x") else "0x" + h

    if isinstance(h, bytes):
        return "0x" + h.hex()

    if hasattr(h, "hex"):
        hx = h.hex()
        return hx if hx.startswith("0x") else "0x" + hx

    raise TypeError(f"Unsupported hash type: {type(h)}")


# =====================================================================================
# 4. Main delivery process
# =====================================================================================

delivery_rows = []
payload_index_rows = []

total_expected_users = 0
total_payloads_created = 0
total_hashes_stored = 0
total_already_existing = 0
total_skipped_no_recommendations = 0
total_failed = 0

for rec_file in RECOMMENDATION_FILES:

    community_id = extract_community_id_from_path(rec_file)

    print("\n" + "-" * 100)
    print(f"Processing recommendation file for community {community_id}")
    print("-" * 100)
    print("File:", rec_file)

    if community_id not in COMMON_COMMUNITIES:
        print(f"[SKIP] community {community_id} is not in COMMON_COMMUNITIES.")
        continue

    # -------------------------------------------------------------------------
    # 4.1 Load and normalize recommendation file
    # -------------------------------------------------------------------------

    recs_raw_df = pd.read_csv(rec_file)
    recs_df = normalize_recommendation_columns(recs_raw_df)

    print("Rows in recommendation file:", len(recs_df))
    print("Users in recommendation file:", recs_df["user"].nunique())

    # -------------------------------------------------------------------------
    # 4.2 Get community members
    # -------------------------------------------------------------------------

    community_member_user_ids = sorted(
        members_df[
            members_df["community_id"].astype(int) == int(community_id)
        ]["user_id"].astype(int).unique().tolist()
    )

    if not community_member_user_ids:
        raise RuntimeError(
            f"No community members found for community_id={community_id}"
        )

    total_expected_users += len(community_member_user_ids)

    print("Community members:", len(community_member_user_ids))

    # -------------------------------------------------------------------------
    # 4.3 Get trusted node A_k
    # -------------------------------------------------------------------------

    trusted_node_user_id = get_trusted_node_user_id_for_community(community_id)
    trusted_node_address = get_trusted_node_address_for_community(community_id)

    print("A_k user_id:", trusted_node_user_id)
    print("A_k address:", trusted_node_address)

    # -------------------------------------------------------------------------
    # 4.4 Process each member
    # -------------------------------------------------------------------------

    for user_id in community_member_user_ids:

        try:
            user_address = user_address_for_dataset_user(user_id)

            # Build Top-N recommendation list
            recommendations = build_user_topn_recommendations(
                recs_df=recs_df,
                user_id=user_id,
                top_n=DELIVERY_TOP_N
            )

            if len(recommendations) == 0:
                total_skipped_no_recommendations += 1

                delivery_rows.append(
                    {
                        "community_id": int(community_id),
                        "user_id": int(user_id),
                        "user_address": user_address,
                        "trusted_node_user_id": int(trusted_node_user_id),
                        "trusted_node_address": trusted_node_address,
                        "delivery_version": int(DELIVERY_VERSION),
                        "top_n": int(DELIVERY_TOP_N),
                        "recommendation_count": 0,
                        "payload_path": "",
                        "recommendation_hash": "",
                        "tx_hash": "",
                        "record_exists_before": "",
                        "record_exists_after": "",
                        "stored_hash_matches": "",
                        "status": "skipped_no_recommendations",
                        "error": ""
                    }
                )

                continue

            # Deterministic nonce for reproducible hashing
            nonce = (
                f"lastfm|community={community_id}|user={user_id}|"
                f"version={DELIVERY_VERSION}|topN={DELIVERY_TOP_N}"
            )

            payload = build_recommendation_payload_for_hash(
                user_address=user_address,
                community_id=community_id,
                version=DELIVERY_VERSION,
                top_n=DELIVERY_TOP_N,
                recommendations=recommendations,
                nonce=nonce
            )

            payload_json = canonical_json(payload)

            recommendation_hash = compute_recommendation_hash(
                user_address=user_address,
                community_id=community_id,
                version=DELIVERY_VERSION,
                top_n=DELIVERY_TOP_N,
                recommendations=recommendations,
                nonce=nonce
            )

            recommendation_hash_hex = hash_to_hex(recommendation_hash)

            # Save off-chain payload
            community_payload_dir = PAYLOAD_DIR / f"community_{community_id}"
            community_payload_dir.mkdir(parents=True, exist_ok=True)

            payload_file = (
                community_payload_dir
                / f"user_{user_id}_top{DELIVERY_TOP_N}_v{DELIVERY_VERSION}.json"
            )

            payload_to_save = {
                "payload": payload,
                "canonical_json": payload_json,
                "recommendation_hash": recommendation_hash_hex,
                "trusted_node_user_id": int(trusted_node_user_id),
                "trusted_node_address": trusted_node_address,
                "source_recommendation_file": str(rec_file),
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            payload_file.write_text(
                json.dumps(payload_to_save, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )

            total_payloads_created += 1

            # Check existing on-chain record before storage
            record_exists_before = contract.functions.recordExists(
                int(community_id),
                Web3.to_checksum_address(user_address),
                int(DELIVERY_VERSION),
                int(DELIVERY_TOP_N)
            ).call()

            if record_exists_before:
                tx_hash = ""
                status = "already_exists"
                total_already_existing += 1
            else:
                tx_hash, sender_address = store_hash_onchain(
                    community_id=community_id,
                    user_address=user_address,
                    version=DELIVERY_VERSION,
                    top_n=DELIVERY_TOP_N,
                    recommendation_hash=recommendation_hash
                )

                if tx_hash is None:
                    status = "already_exists"
                    total_already_existing += 1
                    tx_hash = ""
                else:
                    status = "stored_onchain"
                    total_hashes_stored += 1

            # Verify existence after storage
            record_exists_after = contract.functions.recordExists(
                int(community_id),
                Web3.to_checksum_address(user_address),
                int(DELIVERY_VERSION),
                int(DELIVERY_TOP_N)
            ).call()

            stored_hash_matches = ""

            # Optional verification if contract exposes getRecommendationHash
            if HAS_GET_RECOMMENDATION_HASH and record_exists_after:
                stored_hash = contract.functions.getRecommendationHash(
                    int(community_id),
                    Web3.to_checksum_address(user_address),
                    int(DELIVERY_VERSION),
                    int(DELIVERY_TOP_N)
                ).call()

                stored_hash_hex = hash_to_hex(stored_hash)

                stored_hash_matches = (
                    stored_hash_hex.lower() == recommendation_hash_hex.lower()
                )

                if not stored_hash_matches:
                    raise RuntimeError(
                        f"Stored hash mismatch for community={community_id}, user={user_id}"
                    )

            delivery_rows.append(
                {
                    "community_id": int(community_id),
                    "user_id": int(user_id),
                    "user_address": user_address,
                    "trusted_node_user_id": int(trusted_node_user_id),
                    "trusted_node_address": trusted_node_address,
                    "delivery_version": int(DELIVERY_VERSION),
                    "top_n": int(DELIVERY_TOP_N),
                    "recommendation_count": int(len(recommendations)),
                    "payload_path": str(payload_file),
                    "recommendation_hash": recommendation_hash_hex,
                    "tx_hash": tx_hash,
                    "record_exists_before": bool(record_exists_before),
                    "record_exists_after": bool(record_exists_after),
                    "stored_hash_matches": stored_hash_matches,
                    "status": status,
                    "error": ""
                }
            )

            payload_index_rows.append(
                {
                    "community_id": int(community_id),
                    "user_id": int(user_id),
                    "user_address": user_address,
                    "payload_path": str(payload_file),
                    "recommendation_hash": recommendation_hash_hex,
                    "delivery_version": int(DELIVERY_VERSION),
                    "top_n": int(DELIVERY_TOP_N)
                }
            )

        except Exception as e:
            total_failed += 1

            delivery_rows.append(
                {
                    "community_id": int(community_id),
                    "user_id": int(user_id),
                    "user_address": "",
                    "trusted_node_user_id": int(trusted_node_user_id),
                    "trusted_node_address": trusted_node_address,
                    "delivery_version": int(DELIVERY_VERSION),
                    "top_n": int(DELIVERY_TOP_N),
                    "recommendation_count": "",
                    "payload_path": "",
                    "recommendation_hash": "",
                    "tx_hash": "",
                    "record_exists_before": "",
                    "record_exists_after": "",
                    "stored_hash_matches": "",
                    "status": "failed",
                    "error": str(e)
                }
            )

            print(
                f"[ERROR] community={community_id}, user={user_id}, error={e}"
            )

    # -------------------------------------------------------------------------
    # 4.5 Community progress summary
    # -------------------------------------------------------------------------

    current_df = pd.DataFrame(delivery_rows)
    current_comm_df = current_df[
        current_df["community_id"].astype(int) == int(community_id)
    ]

    print("[COMMUNITY SUMMARY]")
    print(current_comm_df["status"].value_counts().to_string())


# =====================================================================================
# 5. Save reports
# =====================================================================================

delivery_report_df = pd.DataFrame(delivery_rows)
payload_index_df = pd.DataFrame(payload_index_rows)

delivery_report_df.to_csv(
    HASH_STORAGE_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

payload_index_df.to_csv(
    PAYLOAD_INDEX_PATH,
    index=False,
    encoding="utf-8-sig"
)


# =====================================================================================
# 6. Final summary
# =====================================================================================

status_counts = (
    delivery_report_df["status"]
    .value_counts()
    .to_dict()
    if not delivery_report_df.empty
    else {}
)

successful_records = int(
    delivery_report_df[
        delivery_report_df["status"].isin(
            ["stored_onchain", "already_exists"]
        )
    ].shape[0]
)

failed_records = int(
    delivery_report_df[
        delivery_report_df["status"] == "failed"
    ].shape[0]
)

skipped_no_recs = int(
    delivery_report_df[
        delivery_report_df["status"] == "skipped_no_recommendations"
    ].shape[0]
)

final_summary = {
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "contract_name": "RecommendationDelivery",
    "contract_address": loaded_contract_address,
    "chain_id": int(w3.eth.chain_id),
    "owner_account": OWNER_ACCOUNT,
    "delivery_top_n": int(DELIVERY_TOP_N),
    "delivery_version": int(DELIVERY_VERSION),
    "common_communities": [int(x) for x in sorted(COMMON_COMMUNITIES)],
    "common_communities_count": int(len(COMMON_COMMUNITIES)),
    "recommendation_files_count": int(len(RECOMMENDATION_FILES)),
    "total_expected_community_users": int(total_expected_users),
    "total_payloads_created": int(total_payloads_created),
    "total_hashes_stored_new": int(total_hashes_stored),
    "total_already_existing_records": int(total_already_existing),
    "total_successful_records": int(successful_records),
    "total_skipped_no_recommendations": int(skipped_no_recs),
    "total_failed_records": int(failed_records),
    "status_counts": status_counts,
    "hash_storage_report_path": str(HASH_STORAGE_REPORT_PATH),
    "payload_index_path": str(PAYLOAD_INDEX_PATH),
    "payload_dir": str(PAYLOAD_DIR),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

HASH_STORAGE_SUMMARY_PATH.write_text(
    json.dumps(final_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# =====================================================================================
# 7. Print final result
# =====================================================================================

print("\n" + "=" * 100)
print("[HASH STORAGE REPORT]")
print("Report saved to:")
print(HASH_STORAGE_REPORT_PATH)

print("\nPayload index saved to:")
print(PAYLOAD_INDEX_PATH)

print("\nSummary saved to:")
print(HASH_STORAGE_SUMMARY_PATH)

print("\nPayload directory:")
print(PAYLOAD_DIR)

print("\n" + "=" * 100)
print("[FINAL CHECK]")
print("Common communities count:", final_summary["common_communities_count"])
print("Recommendation files count:", final_summary["recommendation_files_count"])
print("Total expected community users:", final_summary["total_expected_community_users"])
print("Total payloads created:", final_summary["total_payloads_created"])
print("Total new hashes stored on-chain:", final_summary["total_hashes_stored_new"])
print("Total already existing records:", final_summary["total_already_existing_records"])
print("Total successful records:", final_summary["total_successful_records"])
print("Total skipped no recommendations:", final_summary["total_skipped_no_recommendations"])
print("Total failed records:", final_summary["total_failed_records"])
print("Status counts:", final_summary["status_counts"])
print("=" * 100)

if final_summary["total_failed_records"] > 0:
    raise RuntimeError(
        "Some recommendation hash storage operations failed. "
        "راجع التقرير لمعرفة التفاصيل."
    )

print("\n[OK] Cell 10 completed.")

Cell 10 — Build Payloads and Store Recommendation Hashes On-Chain
Current DELIVERY_TOP_N: 5
Current DELIVERY_VERSION: 2
Current DELIVERY_OUTPUT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2
Current PAYLOAD_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2\payloads
Contract has getRecommendationHash: True

----------------------------------------------------------------------------------------------------
Processing recommendation file for community 28
----------------------------------------------------------------------------------------------------
File: D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_28\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64

### فائدة Cell 11 ببساطة

هذه الخلية تتحقق أن كل Hash تم تخزينه في العقد يطابق الـ Payload المحفوظ خارج السلسلة.

يعني تفحص:

1. هل ملف Payload موجود؟
2. هل Hash داخل ملف Payload يطابق Hash المحسوب من جديد؟
3. هل السجل موجود فعلًا داخل العقد؟
4. هل Hash الموجود على العقد يطابق Hash الموجود في Payload؟
Cell 11 — Read and Validate Stored Recommendation Hashes

In [18]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 11 — Read and Validate Stored Recommendation Hashes
# =====================================================================================
#
# Purpose:
#   التحقق من سلامة مرحلة إيصال التوصيات المصححة V2.
#
# What this cell validates:
#   1. تقرأ 07_payload_index.csv الخاص بـ DELIVERY_VERSION = 2.
#   2. تقرأ كل Payload محفوظ خارج السلسلة.
#   3. تعيد حساب Hash من الـ Payload.
#   4. تقارن:
#        - hash الموجود في payload file
#        - hash الموجود في payload index
#        - hash المحسوب من جديد
#        - hash المخزن داخل العقد
#
# Important:
#   هذه الخلية لا ترسل transactions.
#   هي قراءة وتحقق فقط.
#
# Expected:
#   Total validated records = 672
#   Failed validations = 0
#
# =====================================================================================


print("=" * 100)
print("Cell 11 — Read and Validate Stored Recommendation Hashes")
print("=" * 100)

print("Current DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("Current DELIVERY_VERSION:", DELIVERY_VERSION)
print("Current DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)

if int(DELIVERY_VERSION) != 2:
    raise RuntimeError(
        "DELIVERY_VERSION is not 2. هذه الخلية يجب أن تتحقق من النسخة المصححة V2."
    )


# =====================================================================================
# 1. Input and output paths
# =====================================================================================

HASH_STORAGE_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_report.csv"
)

PAYLOAD_INDEX_PATH = (
    DELIVERY_OUTPUT_DIR / "07_payload_index.csv"
)

HASH_STORAGE_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_summary.json"
)

VALIDATION_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "08_validate_stored_hashes_report.csv"
)

VALIDATION_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "08_validate_stored_hashes_summary.json"
)


# =====================================================================================
# 2. Validate required files
# =====================================================================================

required_files = [
    HASH_STORAGE_REPORT_PATH,
    PAYLOAD_INDEX_PATH,
    HASH_STORAGE_SUMMARY_PATH
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

print("[OK] Required V2 delivery files exist.")
print("Hash storage report:", HASH_STORAGE_REPORT_PATH)
print("Payload index:", PAYLOAD_INDEX_PATH)
print("Hash storage summary:", HASH_STORAGE_SUMMARY_PATH)


# =====================================================================================
# 3. Load reports
# =====================================================================================

hash_report_df = pd.read_csv(HASH_STORAGE_REPORT_PATH)
payload_index_df = pd.read_csv(PAYLOAD_INDEX_PATH)

with open(HASH_STORAGE_SUMMARY_PATH, "r", encoding="utf-8") as f:
    hash_storage_summary = json.load(f)

print("\n[OK] Reports loaded successfully.")
print("Rows in hash storage report:", len(hash_report_df))
print("Rows in payload index:", len(payload_index_df))

stored_rows = hash_report_df[
    hash_report_df["status"].astype(str) == "stored_onchain"
].copy()

skipped_rows = hash_report_df[
    hash_report_df["status"].astype(str) == "skipped_no_recommendations"
].copy()

failed_rows = hash_report_df[
    hash_report_df["status"].astype(str) == "failed"
].copy()

print("\nStatus counts from hash storage report:")
print(hash_report_df["status"].value_counts().to_string())

if not failed_rows.empty:
    raise RuntimeError(
        "Hash storage report contains failed records. لا تكمل التحقق قبل مراجعتها."
    )

if len(payload_index_df) != len(stored_rows):
    raise RuntimeError(
        f"Payload index rows ({len(payload_index_df)}) do not match stored_onchain rows ({len(stored_rows)})."
    )

print("\n[OK] Payload index count matches stored_onchain count.")


# =====================================================================================
# 4. Helper functions
# =====================================================================================

def hash_to_hex_v11(h) -> str:
    """
    Convert hash value to 0x-prefixed hex string.
    """
    if isinstance(h, str):
        return h if h.startswith("0x") else "0x" + h

    if isinstance(h, bytes):
        return "0x" + h.hex()

    if hasattr(h, "hex"):
        hx = h.hex()
        return hx if hx.startswith("0x") else "0x" + hx

    raise TypeError(f"Unsupported hash type: {type(h)}")


def recompute_hash_from_payload(payload: dict) -> str:
    """
    Recompute keccak256 hash from the payload using canonical JSON.
    """
    msg = canonical_json(payload)
    return hash_to_hex_v11(w3.keccak(text=msg))


def safe_bool(x):
    """
    Convert possible string/bool values to bool.
    """
    if isinstance(x, bool):
        return x

    if pd.isna(x):
        return False

    return str(x).strip().lower() in ["true", "1", "yes"]


# =====================================================================================
# 5. Validate each stored payload against the blockchain
# =====================================================================================

validation_rows = []

for idx, row in payload_index_df.iterrows():

    community_id = int(row["community_id"])
    user_id = int(row["user_id"])
    user_address = Web3.to_checksum_address(row["user_address"])
    payload_path = Path(str(row["payload_path"]))
    index_hash = hash_to_hex_v11(str(row["recommendation_hash"]))
    version = int(row["delivery_version"])
    top_n = int(row["top_n"])

    print(f"\nValidating community={community_id}, user_id={user_id}")

    row_status = "validated"
    error_msg = ""

    payload_exists = payload_path.exists()

    payload_hash = ""
    recomputed_hash = ""
    onchain_hash = ""

    payload_hash_matches_index = False
    recomputed_hash_matches_payload = False
    recomputed_hash_matches_index = False
    onchain_record_exists = False
    onchain_hash_matches_recomputed = False
    final_valid = False

    try:
        if not payload_exists:
            raise FileNotFoundError(f"Payload file not found: {payload_path}")

        with open(payload_path, "r", encoding="utf-8") as f:
            payload_file_data = json.load(f)

        if "payload" not in payload_file_data:
            raise RuntimeError(f"Missing 'payload' key in {payload_path}")

        if "recommendation_hash" not in payload_file_data:
            raise RuntimeError(f"Missing 'recommendation_hash' key in {payload_path}")

        payload = payload_file_data["payload"]
        payload_hash = hash_to_hex_v11(payload_file_data["recommendation_hash"])

        # Recompute from payload
        recomputed_hash = recompute_hash_from_payload(payload)

        payload_hash_matches_index = (
            payload_hash.lower() == index_hash.lower()
        )

        recomputed_hash_matches_payload = (
            recomputed_hash.lower() == payload_hash.lower()
        )

        recomputed_hash_matches_index = (
            recomputed_hash.lower() == index_hash.lower()
        )

        # Check record exists on-chain
        onchain_record_exists = contract.functions.recordExists(
            int(community_id),
            Web3.to_checksum_address(user_address),
            int(version),
            int(top_n)
        ).call()

        if not onchain_record_exists:
            raise RuntimeError(
                f"On-chain record does not exist for community={community_id}, user_id={user_id}"
            )

        # Read hash from contract
        onchain_hash_raw = contract.functions.getRecommendationHash(
            int(community_id),
            Web3.to_checksum_address(user_address),
            int(version),
            int(top_n)
        ).call()

        onchain_hash = hash_to_hex_v11(onchain_hash_raw)

        onchain_hash_matches_recomputed = (
            onchain_hash.lower() == recomputed_hash.lower()
        )

        final_valid = (
            payload_exists
            and payload_hash_matches_index
            and recomputed_hash_matches_payload
            and recomputed_hash_matches_index
            and onchain_record_exists
            and onchain_hash_matches_recomputed
        )

        if not final_valid:
            row_status = "invalid"

    except Exception as e:
        row_status = "failed_validation"
        error_msg = str(e)

    validation_rows.append(
        {
            "community_id": int(community_id),
            "user_id": int(user_id),
            "user_address": user_address,
            "delivery_version": int(version),
            "top_n": int(top_n),
            "payload_path": str(payload_path),
            "payload_exists": bool(payload_exists),
            "payload_hash": payload_hash,
            "index_hash": index_hash,
            "recomputed_hash": recomputed_hash,
            "onchain_hash": onchain_hash,
            "payload_hash_matches_index": bool(payload_hash_matches_index),
            "recomputed_hash_matches_payload": bool(recomputed_hash_matches_payload),
            "recomputed_hash_matches_index": bool(recomputed_hash_matches_index),
            "onchain_record_exists": bool(onchain_record_exists),
            "onchain_hash_matches_recomputed": bool(onchain_hash_matches_recomputed),
            "final_valid": bool(final_valid),
            "status": row_status,
            "error": error_msg
        }
    )

    if row_status != "validated":
        print("[WARNING]", row_status, error_msg)


# =====================================================================================
# 6. Save validation report
# =====================================================================================

validation_report_df = pd.DataFrame(validation_rows)

validation_report_df.to_csv(
    VALIDATION_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)


# =====================================================================================
# 7. Build validation summary
# =====================================================================================

total_records_checked = int(len(validation_report_df))

total_valid = int(
    validation_report_df["final_valid"].astype(bool).sum()
)

total_invalid = int(
    total_records_checked - total_valid
)

status_counts = (
    validation_report_df["status"].value_counts().to_dict()
    if not validation_report_df.empty
    else {}
)

summary = {
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "step": "Cell 11 Hash Validation",
    "contract_name": "RecommendationDelivery",
    "contract_address": loaded_contract_address,
    "chain_id": int(w3.eth.chain_id),
    "delivery_top_n": int(DELIVERY_TOP_N),
    "delivery_version": int(DELIVERY_VERSION),
    "common_communities_count": int(len(COMMON_COMMUNITIES)),
    "stored_records_from_cell10": int(len(stored_rows)),
    "skipped_no_recommendations_from_cell10": int(len(skipped_rows)),
    "total_records_checked": int(total_records_checked),
    "total_valid_records": int(total_valid),
    "total_invalid_records": int(total_invalid),
    "all_payload_files_exist": bool(validation_report_df["payload_exists"].all()),
    "all_payload_hashes_match_index": bool(validation_report_df["payload_hash_matches_index"].all()),
    "all_recomputed_hashes_match_payload": bool(validation_report_df["recomputed_hash_matches_payload"].all()),
    "all_recomputed_hashes_match_index": bool(validation_report_df["recomputed_hash_matches_index"].all()),
    "all_onchain_records_exist": bool(validation_report_df["onchain_record_exists"].all()),
    "all_onchain_hashes_match_recomputed": bool(validation_report_df["onchain_hash_matches_recomputed"].all()),
    "status_counts": status_counts,
    "validation_report_path": str(VALIDATION_REPORT_PATH),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

VALIDATION_SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# =====================================================================================
# 8. Final output
# =====================================================================================

print("\n" + "=" * 100)
print("[HASH VALIDATION REPORT]")
print("Validation report saved to:")
print(VALIDATION_REPORT_PATH)

print("\nValidation summary saved to:")
print(VALIDATION_SUMMARY_PATH)

print("\n" + "=" * 100)
print("[FINAL CHECK]")
print("Stored records from Cell 10:", summary["stored_records_from_cell10"])
print("Skipped no recommendations from Cell 10:", summary["skipped_no_recommendations_from_cell10"])
print("Total records checked:", summary["total_records_checked"])
print("Total valid records:", summary["total_valid_records"])
print("Total invalid records:", summary["total_invalid_records"])
print("All payload files exist:", summary["all_payload_files_exist"])
print("All payload hashes match index:", summary["all_payload_hashes_match_index"])
print("All recomputed hashes match payload:", summary["all_recomputed_hashes_match_payload"])
print("All recomputed hashes match index:", summary["all_recomputed_hashes_match_index"])
print("All on-chain records exist:", summary["all_onchain_records_exist"])
print("All on-chain hashes match recomputed:", summary["all_onchain_hashes_match_recomputed"])
print("Status counts:", summary["status_counts"])
print("=" * 100)

if summary["total_invalid_records"] > 0:
    raise RuntimeError(
        "Some stored recommendation hashes failed validation. راجع تقرير Cell 11."
    )

print("\n[OK] Cell 11 completed.")

Cell 11 — Read and Validate Stored Recommendation Hashes
Current DELIVERY_TOP_N: 5
Current DELIVERY_VERSION: 2
Current DELIVERY_OUTPUT_DIR: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2
[OK] Required V2 delivery files exist.
Hash storage report: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2\07_recommendation_hash_storage_report.csv
Payload index: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2\07_payload_index.csv
Hash storage summary: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد\03_outputs\RECOMMENDATION_DELIVERY_TOP5_V2\07_recommendation_hash_storage_summary.json

[OK] Reports loaded successfully.
Rows in hash storage report: 673
Rows in payload index: 672

Status counts from hash storage report:
status
s

### الكود كامل

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 1 — Paths, Imports, and File Checks
# =====================================================================================
#
# Purpose:
#   تعريف المسارات الأساسية، تحميل المكتبات، تحديد المجتمعات المشتركة،
#   وفحص وجود ملفات الإعلان وملفات توصيات Top-20.
#
# =====================================================================================

import os
import re
import json
import pandas as pd

from pathlib import Path
from datetime import datetime
from web3 import Web3


# =====================================================================================
# 1. Base paths
# =====================================================================================

DELIVERY_BASE_DIR = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\4 ايصال التوصيات بضمنها العقد"
)

ANNOUNCEMENT_BASE_DIR = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

COMMON_INPUT_DIR = DELIVERY_BASE_DIR / "00_common_inputs"
CONTRACT_DIR = DELIVERY_BASE_DIR / "01_contract"
PYTHON_SCRIPTS_DIR = DELIVERY_BASE_DIR / "02_Python_scripts"
OUTPUTS_DIR = DELIVERY_BASE_DIR / "03_outputs"

for d in [COMMON_INPUT_DIR, CONTRACT_DIR, PYTHON_SCRIPTS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# 2. Announcement-stage files
# =====================================================================================

ANNOUNCEMENT_LOGS_DIR = ANNOUNCEMENT_BASE_DIR / "outputs" / "logs"

ADDRESS_TO_USERID_PATH = ANNOUNCEMENT_LOGS_DIR / "address_to_userid.csv"
COMMUNITY_MEMBERS_PATH = ANNOUNCEMENT_LOGS_DIR / "community_members.csv"
OWNER_ADDRESS_PATH = ANNOUNCEMENT_LOGS_DIR / "owner_address.txt"

LOAD_MEMBERS_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "load_members_report.csv"
ANNOUNCE_AK_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "announce_ak_report.csv"
CONFIRM_AK_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "confirm_ak_report.csv"
FINALIZED_AK_REPORT_PATH = ANNOUNCEMENT_LOGS_DIR / "finalized_ak_report.csv"


# =====================================================================================
# 3. Common Last.fm communities
# =====================================================================================

COMMON_COMMUNITIES = [
    28, 30, 38, 40, 44, 60, 99,
    104, 107, 114, 204, 211, 506
]


# =====================================================================================
# 4. Last.fm Top-20 recommendation files for common communities
# =====================================================================================

LASTFM_COMMON_TOP20_RECS_FILES = {
    28: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_28\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM28.csv",
    30: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_30\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM30\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM30.csv",
    38: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_38\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM38\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM38.csv",
    40: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_40\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM40\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM40.csv",
    44: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_44\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM44\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM44.csv",
    60: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_60\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM60\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM60.csv",
    99: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_99\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM99\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM99.csv",
    104: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_104\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM104\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM104.csv",
    107: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_107\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM107\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM107.csv",
    114: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_114\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM114\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM114.csv",
    204: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_204\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM204\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM204.csv",
    211: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_211\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM211\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM211.csv",
    506: r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Uplift Recommendation\LastFM_Discovered_COMM_alpha_0.340\DMF_COMMUNITIES_VR14b_SAMPLE99_ONLY\COMM_506\RUN_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM506\LastFM_recs_top20_VR14b_COMM_LN0d4_NEG1_EMB64_HL64_POS1_BPR_NNEG99_COMM506.csv",
}


# =====================================================================================
# 5. Validate required files
# =====================================================================================

print("=" * 100)
print("Cell 1 — Paths, Imports, and File Checks")
print("=" * 100)

required_announcement_files = [
    ADDRESS_TO_USERID_PATH,
    COMMUNITY_MEMBERS_PATH,
    OWNER_ADDRESS_PATH,
    LOAD_MEMBERS_REPORT_PATH,
    ANNOUNCE_AK_REPORT_PATH,
    CONFIRM_AK_REPORT_PATH,
    FINALIZED_AK_REPORT_PATH,
]

missing_required = [p for p in required_announcement_files if not p.exists()]

if missing_required:
    raise FileNotFoundError(
        "Missing required announcement files:\n"
        + "\n".join(str(p) for p in missing_required)
    )

print("[OK] All required announcement files exist.")

missing_recs = []

for cid, p in LASTFM_COMMON_TOP20_RECS_FILES.items():
    path_obj = Path(p)

    if not path_obj.exists():
        missing_recs.append((cid, path_obj))

if missing_recs:
    raise FileNotFoundError(
        "Missing recommendation files:\n"
        + "\n".join(f"{cid}: {p}" for cid, p in missing_recs)
    )

print("[OK] All recommendation files exist.")
print("Common communities:", COMMON_COMMUNITIES)
print("Number of common communities:", len(COMMON_COMMUNITIES))

print("\n[OK] Cell 1 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 2 — Create Common-Community Input Files
# =====================================================================================
#
# Purpose:
#   إنشاء ملفات إدخال خاصة بالمجتمعات المشتركة فقط.
#
# Output:
#   00_common_inputs
#
# =====================================================================================

print("=" * 100)
print("Cell 2 — Create Common-Community Input Files")
print("=" * 100)


def pick_col(df: pd.DataFrame, candidates: list, required: bool = True):
    """
    Pick the first existing column from candidates.
    """
    lower_to_original = {
        str(c).lower().strip(): c
        for c in df.columns
    }

    for c in candidates:
        key = str(c).lower().strip()

        if key in lower_to_original:
            return lower_to_original[key]

    if required:
        raise ValueError(
            f"Cannot find any of columns {candidates}. Available: {list(df.columns)}"
        )

    return None


def filter_by_common_communities(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Filter a report by common communities using community_id column.
    """
    cid_col = pick_col(
        df,
        ["community_id", "community", "communityId", "cid"]
    )

    out = df.copy()
    out[cid_col] = out[cid_col].astype(int)

    out = out[
        out[cid_col].isin(COMMON_COMMUNITIES)
    ].copy()

    if cid_col != "community_id":
        out = out.rename(columns={cid_col: "community_id"})

    out = out.sort_values("community_id").reset_index(drop=True)

    print(f"[OK] {name}: {len(out)} rows after filtering common communities.")

    return out


# =====================================================================================
# 1. Load announcement reports
# =====================================================================================

load_members_df = pd.read_csv(LOAD_MEMBERS_REPORT_PATH)
announce_df = pd.read_csv(ANNOUNCE_AK_REPORT_PATH)
confirm_df = pd.read_csv(CONFIRM_AK_REPORT_PATH)
finalized_df = pd.read_csv(FINALIZED_AK_REPORT_PATH)

print("[OK] Announcement reports loaded.")


# =====================================================================================
# 2. Filter to common communities
# =====================================================================================

load_members_common_df = filter_by_common_communities(
    load_members_df,
    "load_members_report"
)

announce_common_df = filter_by_common_communities(
    announce_df,
    "announce_ak_report"
)

confirm_common_df = filter_by_common_communities(
    confirm_df,
    "confirm_ak_report"
)

finalized_common_df = filter_by_common_communities(
    finalized_df,
    "finalized_ak_report"
)


# =====================================================================================
# 3. Save filtered reports
# =====================================================================================

LOAD_MEMBERS_COMMON_PATH = (
    COMMON_INPUT_DIR / "load_members_report_common_communities.csv"
)

ANNOUNCE_COMMON_PATH = (
    COMMON_INPUT_DIR / "announce_ak_report_common_communities.csv"
)

CONFIRM_COMMON_PATH = (
    COMMON_INPUT_DIR / "confirm_ak_report_common_communities.csv"
)

FINALIZED_COMMON_PATH = (
    COMMON_INPUT_DIR / "finalized_ak_report_common_communities.csv"
)

EXCLUDED_COMMUNITIES_PATH = (
    COMMON_INPUT_DIR / "lastfm_excluded_announcement_communities.csv"
)

COMMON_MANIFEST_PATH = (
    COMMON_INPUT_DIR / "lastfm_common_communities_manifest.csv"
)

load_members_common_df.to_csv(
    LOAD_MEMBERS_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

announce_common_df.to_csv(
    ANNOUNCE_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

confirm_common_df.to_csv(
    CONFIRM_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

finalized_common_df.to_csv(
    FINALIZED_COMMON_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("[OK] Common-community reports saved.")


# =====================================================================================
# 4. Excluded announcement communities
# =====================================================================================

finalized_cid_col = pick_col(
    finalized_df,
    ["community_id", "community", "communityId", "cid"]
)

all_finalized_communities = sorted(
    finalized_df[finalized_cid_col].astype(int).unique().tolist()
)

excluded_communities = sorted(
    set(all_finalized_communities) - set(COMMON_COMMUNITIES)
)

excluded_df = pd.DataFrame({
    "excluded_announcement_community": excluded_communities,
    "reason": "No matching Top-20 recommendation file in the common delivery set"
})

excluded_df.to_csv(
    EXCLUDED_COMMUNITIES_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("[OK] Excluded communities saved.")
print("Excluded communities:", excluded_communities)


# =====================================================================================
# 5. Build manifest
# =====================================================================================

cid_col = pick_col(
    finalized_common_df,
    ["community_id"]
)

ak_col = pick_col(
    finalized_common_df,
    [
        "finalized_Ak",
        "finalized_ak",
        "A_k",
        "Ak",
        "authorized_node",
        "trusted_node_user_id"
    ]
)

community_size_col = pick_col(
    finalized_common_df,
    ["community_size", "size", "member_count", "members_count"],
    required=False
)

accepted_col = pick_col(
    finalized_common_df,
    [
        "accepted_confirmations",
        "confirmations_received",
        "accepted_count",
        "confirmation_count"
    ],
    required=False
)

required_col = pick_col(
    finalized_common_df,
    [
        "required_confirmations",
        "required_count",
        "majority_required"
    ],
    required=False
)

manifest_rows = []

for _, row in finalized_common_df.iterrows():

    cid = int(row[cid_col])
    ak = int(row[ak_col])

    recommendation_file = Path(
        LASTFM_COMMON_TOP20_RECS_FILES[cid]
    )

    community_size = (
        int(row[community_size_col])
        if community_size_col is not None and pd.notna(row[community_size_col])
        else ""
    )

    accepted_confirmations = (
        int(row[accepted_col])
        if accepted_col is not None and pd.notna(row[accepted_col])
        else ""
    )

    required_confirmations = (
        int(row[required_col])
        if required_col is not None and pd.notna(row[required_col])
        else ""
    )

    manifest_rows.append(
        {
            "community_id": cid,
            "finalized_Ak": ak,
            "community_size": community_size,
            "accepted_confirmations": accepted_confirmations,
            "required_confirmations": required_confirmations,
            "validation_status": "valid_finalized",
            "recommendation_file": str(recommendation_file),
            "recommendation_file_exists": bool(recommendation_file.exists()),
            "included_in_delivery": bool(True)
        }
    )

manifest_df = pd.DataFrame(manifest_rows)
manifest_df = manifest_df.sort_values("community_id").reset_index(drop=True)

manifest_df.to_csv(
    COMMON_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n[COMMON MANIFEST]")
print(manifest_df.to_string(index=False))

if not manifest_df["recommendation_file_exists"].all():
    raise FileNotFoundError(
        "Some recommendation files in manifest do not exist."
    )

if sorted(manifest_df["community_id"].tolist()) != sorted(COMMON_COMMUNITIES):
    raise RuntimeError(
        "Manifest communities do not match COMMON_COMMUNITIES."
    )

print("\n[FINAL CHECK]")
print("All common communities are valid_finalized and have Top-20 recommendation files.")
print("Common manifest saved to:", COMMON_MANIFEST_PATH)

print("\n[OK] Cell 2 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 3 — Write RecommendationDelivery Solidity Smart Contract
# =====================================================================================
#
# Purpose:
#   إنشاء عقد RecommendationDelivery.
#
# Main design:
#   - Stores only recommendation hash.
#   - Full payload stays off-chain.
#   - Only finalized A_k can store hashes for its community.
#   - Target user must be registered as a member of that community.
#
# =====================================================================================

print("=" * 100)
print("Cell 3 — Write RecommendationDelivery Solidity Smart Contract")
print("=" * 100)

SOLIDITY_FILE = CONTRACT_DIR / "RecommendationDelivery.sol"

solidity_source = r'''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract RecommendationDelivery {

    address public owner;

    mapping(uint256 => address) public authorizedNode;

    mapping(uint256 => mapping(address => bool)) public isCommunityMember;

    mapping(uint256 => uint256) public communityMemberCount;

    mapping(bytes32 => bytes32) private recommendationHashByKey;

    mapping(bytes32 => bool) private recommendationRecordExists;

    event AuthorizedNodeSet(
        uint256 indexed communityId,
        address indexed node
    );

    event CommunityMemberRegistered(
        uint256 indexed communityId,
        address indexed member
    );

    event RecommendationHashStored(
        uint256 indexed communityId,
        address indexed userAddress,
        uint256 indexed version,
        uint256 topN,
        bytes32 recommendationHash,
        address sender
    );

    modifier onlyOwner() {
        require(msg.sender == owner, "Only owner can call this function");
        _;
    }

    modifier onlyAuthorizedNode(uint256 communityId) {
        require(
            authorizedNode[communityId] == msg.sender,
            "Only authorized node can store hashes for this community"
        );
        _;
    }

    constructor() {
        owner = msg.sender;
    }

    function setAuthorizedNode(
        uint256 communityId,
        address node
    )
        external
        onlyOwner
    {
        require(node != address(0), "Invalid node address");

        address current = authorizedNode[communityId];

        require(
            current == address(0) || current == node,
            "Authorized node already set to different address"
        );

        if (current == node) {
            return;
        }

        authorizedNode[communityId] = node;

        emit AuthorizedNodeSet(communityId, node);
    }

    function registerCommunityMembers(
        uint256 communityId,
        address[] calldata members
    )
        external
        onlyOwner
    {
        for (uint256 i = 0; i < members.length; i++) {

            address member = members[i];

            require(member != address(0), "Invalid member address");

            if (!isCommunityMember[communityId][member]) {

                isCommunityMember[communityId][member] = true;

                communityMemberCount[communityId] += 1;

                emit CommunityMemberRegistered(communityId, member);
            }
        }
    }

    function _recordKey(
        uint256 communityId,
        address userAddress,
        uint256 version,
        uint256 topN
    )
        internal
        pure
        returns (bytes32)
    {
        return keccak256(
            abi.encodePacked(
                communityId,
                userAddress,
                version,
                topN
            )
        );
    }

    function storeRecommendationHash(
        uint256 communityId,
        address userAddress,
        uint256 version,
        uint256 topN,
        bytes32 recommendationHash
    )
        external
        onlyAuthorizedNode(communityId)
    {
        require(userAddress != address(0), "Invalid user address");

        require(
            isCommunityMember[communityId][userAddress],
            "Target user is not a registered community member"
        );

        bytes32 key = _recordKey(
            communityId,
            userAddress,
            version,
            topN
        );

        require(
            !recommendationRecordExists[key],
            "Recommendation hash already stored"
        );

        recommendationHashByKey[key] = recommendationHash;
        recommendationRecordExists[key] = true;

        emit RecommendationHashStored(
            communityId,
            userAddress,
            version,
            topN,
            recommendationHash,
            msg.sender
        );
    }

    function recordExists(
        uint256 communityId,
        address userAddress,
        uint256 version,
        uint256 topN
    )
        external
        view
        returns (bool)
    {
        bytes32 key = _recordKey(
            communityId,
            userAddress,
            version,
            topN
        );

        return recommendationRecordExists[key];
    }

    function getRecommendationHash(
        uint256 communityId,
        address userAddress,
        uint256 version,
        uint256 topN
    )
        external
        view
        returns (bytes32)
    {
        bytes32 key = _recordKey(
            communityId,
            userAddress,
            version,
            topN
        );

        require(
            recommendationRecordExists[key],
            "Recommendation hash does not exist"
        );

        return recommendationHashByKey[key];
    }
}
'''

SOLIDITY_FILE.write_text(
    solidity_source,
    encoding="utf-8"
)

print("[OK] Solidity contract saved to:")
print(SOLIDITY_FILE)

print("\nContract name: RecommendationDelivery")
print("Design: stores recommendation hash only; payload remains off-chain.")

print("\n[OK] Cell 3 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 4 — Connect to Ganache
# =====================================================================================

RPC_URL = "http://127.0.0.1:7545"

w3 = Web3(Web3.HTTPProvider(RPC_URL))

if not w3.is_connected():
    raise RuntimeError(
        "Cannot connect to Ganache. تأكد أن Ganache شغال على http://127.0.0.1:7545"
    )

accounts = w3.eth.accounts

if len(accounts) < 2:
    raise RuntimeError("Ganache must provide at least two accounts.")

OWNER_ACCOUNT = Web3.to_checksum_address(accounts[0])

owner_from_announcement = Web3.to_checksum_address(
    OWNER_ADDRESS_PATH.read_text(encoding="utf-8").strip()
)

print("=" * 100)
print("Cell 4 — Connect to Ganache")
print("=" * 100)

print("Connected to Ganache:", w3.is_connected())
print("RPC URL:", RPC_URL)
print("Chain ID:", w3.eth.chain_id)
print("Number of accounts:", len(accounts))
print("Owner Account from Ganache:", OWNER_ACCOUNT)
print("Owner Account from Announcement:", owner_from_announcement)

if OWNER_ACCOUNT.lower() != owner_from_announcement.lower():
    raise RuntimeError(
        "Owner account mismatch. يبدو أن Ganache تغير أو أُعيد تشغيله بحسابات مختلفة."
    )

print("\nFirst Ganache accounts:")
for i, acc in enumerate(accounts[:10]):
    print(i, acc)

print("\n[FINAL CHECK]")
print("Owner account matches the announcement-stage owner.")
print("Ganache environment is consistent with the previous Last.fm announcement stage.")

print("\n[OK] Cell 4 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 5 — Compile and Deploy RecommendationDelivery Smart Contract
# =====================================================================================

from solcx import compile_standard, install_solc, set_solc_version

SOLC_VERSION = "0.8.20"

ABI_OUTPUT_PATH = CONTRACT_DIR / "RecommendationDelivery_abi.json"
ADDRESS_OUTPUT_PATH = CONTRACT_DIR / "contract_address.txt"
DEPLOYMENT_LOG_PATH = OUTPUTS_DIR / "04_smart_contract_deployment_log.json"

print("=" * 100)
print("Cell 5 — Compile and Deploy RecommendationDelivery Smart Contract")
print("=" * 100)

print("Installing / selecting Solidity compiler:", SOLC_VERSION)

install_solc(SOLC_VERSION)
set_solc_version(SOLC_VERSION)

if not SOLIDITY_FILE.exists():
    raise FileNotFoundError(f"Solidity file not found: {SOLIDITY_FILE}")

source_code = SOLIDITY_FILE.read_text(encoding="utf-8")

print("\n[OK] Solidity source loaded from:")
print(SOLIDITY_FILE)

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {
            "RecommendationDelivery.sol": {
                "content": source_code
            }
        },
        "settings": {
            "optimizer": {
                "enabled": True,
                "runs": 200
            },
            "outputSelection": {
                "*": {
                    "*": [
                        "abi",
                        "evm.bytecode"
                    ]
                }
            }
        }
    }
)

contract_data = compiled["contracts"]["RecommendationDelivery.sol"]["RecommendationDelivery"]

contract_abi = contract_data["abi"]
contract_bytecode = contract_data["evm"]["bytecode"]["object"]

if not contract_bytecode:
    raise RuntimeError("Compiled bytecode is empty. Contract compilation failed.")

print("\n[OK] Compilation completed successfully.")
print("Contract name: RecommendationDelivery")

RecommendationDelivery = w3.eth.contract(
    abi=contract_abi,
    bytecode=contract_bytecode
)

print("\nDeploying RecommendationDelivery contract...")

deploy_tx_hash = RecommendationDelivery.constructor().transact(
    {
        "from": OWNER_ACCOUNT,
        "gas": 7_000_000
    }
)

deploy_receipt = w3.eth.wait_for_transaction_receipt(deploy_tx_hash)

if deploy_receipt.status != 1:
    raise RuntimeError("Contract deployment failed.")

contract_address = Web3.to_checksum_address(deploy_receipt.contractAddress)

print("\n[OK] Contract deployed successfully.")
print("Contract address:", contract_address)
print("Deployment tx:", deploy_tx_hash.hex())

ABI_OUTPUT_PATH.write_text(
    json.dumps(contract_abi, indent=2),
    encoding="utf-8"
)

ADDRESS_OUTPUT_PATH.write_text(
    contract_address,
    encoding="utf-8"
)

deployment_log = {
    "deployment_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "rpc_url": RPC_URL,
    "chain_id": w3.eth.chain_id,
    "contract_name": "RecommendationDelivery",
    "contract_address": contract_address,
    "deployment_tx_hash": deploy_tx_hash.hex(),
    "owner_account": OWNER_ACCOUNT,
    "solidity_file": str(SOLIDITY_FILE),
    "abi_path": str(ABI_OUTPUT_PATH),
    "contract_address_path": str(ADDRESS_OUTPUT_PATH),
    "solc_version": SOLC_VERSION,
    "ganache_accounts_count": len(accounts),
    "common_communities_count": len(COMMON_COMMUNITIES),
    "common_communities": COMMON_COMMUNITIES,
}

DEPLOYMENT_LOG_PATH.write_text(
    json.dumps(deployment_log, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[OK] ABI saved to:")
print(ABI_OUTPUT_PATH)

print("\n[OK] Contract address saved to:")
print(ADDRESS_OUTPUT_PATH)

print("\n[OK] Deployment log saved to:")
print(DEPLOYMENT_LOG_PATH)

print("\n[OK] Cell 5 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 6 — Load RecommendationDelivery Contract and Common Last.fm Data
# =====================================================================================

print("=" * 100)
print("Cell 6 — Load RecommendationDelivery Contract and Common Last.fm Data")
print("=" * 100)

if not ABI_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"ABI file not found: {ABI_OUTPUT_PATH}")

if not ADDRESS_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Contract address file not found: {ADDRESS_OUTPUT_PATH}")

with open(ABI_OUTPUT_PATH, "r", encoding="utf-8") as f:
    loaded_abi = json.load(f)

loaded_contract_address = Web3.to_checksum_address(
    ADDRESS_OUTPUT_PATH.read_text(encoding="utf-8").strip()
)

contract = w3.eth.contract(
    address=loaded_contract_address,
    abi=loaded_abi
)

print("[OK] Loaded RecommendationDelivery contract at:")
print(loaded_contract_address)

contract_owner = Web3.to_checksum_address(
    contract.functions.owner().call()
)

print("\nContract owner:", contract_owner)
print("Expected owner:", OWNER_ACCOUNT)

if contract_owner.lower() != OWNER_ACCOUNT.lower():
    raise RuntimeError(
        "Contract owner does not match OWNER_ACCOUNT."
    )

print("[OK] Contract owner matches OWNER_ACCOUNT.")

for p in [
    ADDRESS_TO_USERID_PATH,
    COMMUNITY_MEMBERS_PATH,
    COMMON_MANIFEST_PATH,
    FINALIZED_COMMON_PATH
]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

address_df = pd.read_csv(ADDRESS_TO_USERID_PATH)
members_df = pd.read_csv(COMMUNITY_MEMBERS_PATH)
manifest_df = pd.read_csv(COMMON_MANIFEST_PATH)
finalized_common_df = pd.read_csv(FINALIZED_COMMON_PATH)

print("\n[OK] Input files loaded successfully.")

address_df["user_id"] = address_df["user_id"].astype(int)
address_df["address"] = address_df["address"].astype(str)

members_df["community_id"] = members_df["community_id"].astype(int)
members_df["user_id"] = members_df["user_id"].astype(int)

manifest_df["community_id"] = manifest_df["community_id"].astype(int)
manifest_df["finalized_Ak"] = manifest_df["finalized_Ak"].astype(int)

finalized_common_df["community_id"] = finalized_common_df["community_id"].astype(int)

print("[OK] Data types normalized.")

manifest_communities = sorted(
    manifest_df["community_id"].astype(int).unique().tolist()
)

if manifest_communities != sorted(COMMON_COMMUNITIES):
    raise RuntimeError(
        "Manifest communities do not match COMMON_COMMUNITIES.\n"
        f"Manifest: {manifest_communities}\n"
        f"Expected: {sorted(COMMON_COMMUNITIES)}"
    )

print("\n[OK] Manifest communities match COMMON_COMMUNITIES.")
print("Common communities:", manifest_communities)

if "validation_status" in manifest_df.columns:
    invalid_status = manifest_df[
        manifest_df["validation_status"] != "valid_finalized"
    ]

    if not invalid_status.empty:
        raise RuntimeError(
            "Some manifest communities are not valid_finalized:\n"
            + invalid_status.to_string(index=False)
        )

print("[OK] All manifest communities are valid_finalized.")

if "recommendation_file_exists" in manifest_df.columns:
    if not manifest_df["recommendation_file_exists"].astype(bool).all():
        raise RuntimeError(
            "Some recommendation files are marked as missing in the manifest."
        )

print("[OK] All manifest communities have recommendation files.")

user_to_address = {
    int(row["user_id"]): Web3.to_checksum_address(row["address"])
    for _, row in address_df.iterrows()
}

print("\nTotal mapped user IDs:", len(user_to_address))

trusted_nodes_df = manifest_df[
    ["community_id", "finalized_Ak"]
].copy()

trusted_nodes_df = trusted_nodes_df.rename(
    columns={
        "finalized_Ak": "trusted_node_user_id"
    }
)

trusted_nodes_df["trusted_node_address"] = trusted_nodes_df["trusted_node_user_id"].map(
    user_to_address
)

if trusted_nodes_df["trusted_node_address"].isna().any():
    missing = trusted_nodes_df[
        trusted_nodes_df["trusted_node_address"].isna()
    ]

    raise RuntimeError(
        "Missing blockchain addresses for some finalized A_k values:\n"
        + missing.to_string(index=False)
    )

trusted_nodes_df["trusted_node_address"] = trusted_nodes_df["trusted_node_address"].apply(
    Web3.to_checksum_address
)

print("\n[OK] Trusted-node addresses mapped from finalized A_k values.")

ganache_accounts_lower = set(
    Web3.to_checksum_address(a).lower()
    for a in accounts
)

missing_unlocked_accounts = []

for _, row in trusted_nodes_df.iterrows():
    cid = int(row["community_id"])
    ak_user_id = int(row["trusted_node_user_id"])
    ak_address = Web3.to_checksum_address(row["trusted_node_address"])

    if ak_address.lower() not in ganache_accounts_lower:
        missing_unlocked_accounts.append(
            {
                "community_id": cid,
                "trusted_node_user_id": ak_user_id,
                "trusted_node_address": ak_address
            }
        )

if missing_unlocked_accounts:
    missing_df = pd.DataFrame(missing_unlocked_accounts)

    raise RuntimeError(
        "Some A_k addresses are not available as unlocked Ganache accounts:\n"
        + missing_df.to_string(index=False)
    )

print("[OK] All A_k addresses are available as unlocked Ganache accounts.")

TRUSTED_NODES_OUTPUT_PATH = (
    OUTPUTS_DIR / "05_lastfm_trusted_nodes_common_from_finalized_announcement.csv"
)

trusted_nodes_df.to_csv(
    TRUSTED_NODES_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nTrusted nodes for common communities:")
print(trusted_nodes_df.to_string(index=False))

print("\n[OK] Trusted-node mapping saved to:")
print(TRUSTED_NODES_OUTPUT_PATH)

print("\n[FINAL SUMMARY]")
print("Loaded contract:", loaded_contract_address)
print("Contract owner:", contract_owner)
print("Common communities count:", len(COMMON_COMMUNITIES))
print("Trusted nodes count:", len(trusted_nodes_df))
print("Total mapped user IDs:", len(user_to_address))

print("\n[OK] Cell 6 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 7 — Final Corrected Delivery Configuration V2
# =====================================================================================
#
# Purpose:
#   استخدام النسخة النهائية المصححة.
#
# Important:
#   لا نعتمد V1.
#   النسخة البحثية المعتمدة هي V2.
#
# =====================================================================================

print("=" * 100)
print("Cell 7 — Final Corrected Delivery Configuration V2")
print("=" * 100)

DELIVERY_TOP_N = 5
DELIVERY_VERSION = 2

DELIVERY_OUTPUT_DIR = (
    OUTPUTS_DIR
    / f"RECOMMENDATION_DELIVERY_TOP{DELIVERY_TOP_N}_V{DELIVERY_VERSION}"
)

PAYLOAD_DIR = DELIVERY_OUTPUT_DIR / "payloads"

DELIVERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("DELIVERY_VERSION:", DELIVERY_VERSION)
print("DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)
print("PAYLOAD_DIR:", PAYLOAD_DIR)

if "included_in_delivery" not in manifest_df.columns:
    raise RuntimeError(
        "Column included_in_delivery not found in manifest_df."
    )

if "recommendation_file" not in manifest_df.columns:
    raise RuntimeError(
        "Column recommendation_file not found in manifest_df."
    )

RECOMMENDATION_FILES = [
    Path(row["recommendation_file"])
    for _, row in manifest_df.sort_values("community_id").iterrows()
    if bool(row["included_in_delivery"])
]

print("\nRecommendation files used in delivery:")

missing_files = []

for p in RECOMMENDATION_FILES:
    exists = p.exists()
    print(exists, "=>", p)

    if not exists:
        missing_files.append(p)

if missing_files:
    raise FileNotFoundError(
        "Some recommendation files are missing:\n"
        + "\n".join(str(p) for p in missing_files)
    )

if len(RECOMMENDATION_FILES) != len(COMMON_COMMUNITIES):
    raise RuntimeError(
        f"Expected {len(COMMON_COMMUNITIES)} recommendation files, "
        f"but got {len(RECOMMENDATION_FILES)}"
    )

print("\n[FINAL CHECK]")
print("All common-community recommendation files are ready for corrected V2 delivery.")
print("Number of recommendation files:", len(RECOMMENDATION_FILES))
print("Number of common communities:", len(COMMON_COMMUNITIES))

print("\n[OK] Cell 7 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 8 — Helper Functions with Corrected Last.fm Mapping
# =====================================================================================
#
# Critical correction:
#   userID   -> user
#   artistID -> item
#
# We do NOT use:
#   user  -> because it is a local index
#   i_idx -> because it is a local item index
#
# =====================================================================================


def extract_community_id_from_path(path: Path) -> int:
    """
    Extract community_id from recommendation file name.
    """
    file_name = path.name

    patterns = [
        r"comm_(\d+)",
        r"COMM_(\d+)",
        r"COMM(\d+)",
        r"community_(\d+)",
        r"Community_(\d+)",
    ]

    for pat in patterns:
        match = re.search(pat, file_name)

        if match:
            return int(match.group(1))

    raise ValueError(
        f"Cannot extract community_id from file name: {file_name}"
    )


def get_trusted_node_address_for_community(community_id: int) -> str:
    """
    Return blockchain address of finalized A_k for a given community.
    """
    row = trusted_nodes_df[
        trusted_nodes_df["community_id"] == int(community_id)
    ]

    if row.empty:
        raise ValueError(
            f"No trusted node found for community_id={community_id}"
        )

    return Web3.to_checksum_address(
        row.iloc[0]["trusted_node_address"]
    )


def get_trusted_node_user_id_for_community(community_id: int) -> int:
    """
    Return user_id of finalized A_k for a given community.
    """
    row = trusted_nodes_df[
        trusted_nodes_df["community_id"] == int(community_id)
    ]

    if row.empty:
        raise ValueError(
            f"No trusted node found for community_id={community_id}"
        )

    return int(row.iloc[0]["trusted_node_user_id"])


def user_address_for_dataset_user(user_id: int) -> str:
    """
    Convert Last.fm original user_id into blockchain address.
    """
    user_id = int(user_id)

    if user_id not in user_to_address:
        raise ValueError(
            f"user_id={user_id} does not exist in Last.fm address_to_userid.csv"
        )

    return Web3.to_checksum_address(
        user_to_address[user_id]
    )


def build_recommendation_payload_for_hash(
    user_address: str,
    community_id: int,
    version: int,
    top_n: int,
    recommendations: list,
    nonce: str
) -> dict:
    """
    Build payload used for hash computation.
    """
    return {
        "dataset": "Last.fm",
        "user_address": Web3.to_checksum_address(user_address),
        "community_id": int(community_id),
        "version": int(version),
        "top_n": int(top_n),
        "recommendations": recommendations,
        "nonce": str(nonce)
    }


def canonical_json(data: dict) -> str:
    """
    Deterministic JSON representation.
    """
    return json.dumps(
        data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )


def compute_recommendation_hash(
    user_address: str,
    community_id: int,
    version: int,
    top_n: int,
    recommendations: list,
    nonce: str
):
    """
    Compute keccak256 hash for the recommendation payload.
    """
    data = build_recommendation_payload_for_hash(
        user_address=user_address,
        community_id=community_id,
        version=version,
        top_n=top_n,
        recommendations=recommendations,
        nonce=nonce
    )

    message = canonical_json(data)

    return w3.keccak(text=message)


def ensure_authorized_node(community_id: int):
    """
    Register finalized A_k inside RecommendationDelivery if needed.
    """
    trusted_node_address = get_trusted_node_address_for_community(
        community_id
    )

    current_node = contract.functions.authorizedNode(
        int(community_id)
    ).call()

    if current_node.lower() == trusted_node_address.lower():
        print(
            f"[A_k] Already set for community {community_id}: "
            f"{trusted_node_address}"
        )

        return trusted_node_address, None

    tx_hash = contract.functions.setAuthorizedNode(
        int(community_id),
        trusted_node_address
    ).transact(
        {
            "from": OWNER_ACCOUNT,
            "gas": 500_000
        }
    )

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError(
            f"Failed to set A_k for community {community_id}"
        )

    print(
        f"[A_k] Set for community {community_id}: "
        f"{trusted_node_address}"
    )

    return trusted_node_address, tx_hash.hex()


def register_community_members_onchain(
    community_id: int,
    member_addresses: list,
    batch_size: int = 100
):
    """
    Register community members inside RecommendationDelivery.
    """
    member_addresses = [
        Web3.to_checksum_address(x)
        for x in member_addresses
    ]

    unique_members = sorted(set(member_addresses))

    print(
        f"[MEMBERS] Registering {len(unique_members)} members "
        f"for community {community_id}"
    )

    tx_hashes = []

    current_count_before = int(
        contract.functions.communityMemberCount(
            int(community_id)
        ).call()
    )

    for start in range(0, len(unique_members), batch_size):
        batch = unique_members[start:start + batch_size]

        tx_hash = contract.functions.registerCommunityMembers(
            int(community_id),
            batch
        ).transact(
            {
                "from": OWNER_ACCOUNT,
                "gas": 6_000_000
            }
        )

        receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

        if receipt.status != 1:
            raise RuntimeError(
                f"Failed to register members for community={community_id}, "
                f"batch_start={start}"
            )

        tx_hashes.append(tx_hash.hex())

        print(
            f"[MEMBERS] community={community_id}, "
            f"batch={start // batch_size + 1}, "
            f"tx={tx_hash.hex()}"
        )

    current_count_after = int(
        contract.functions.communityMemberCount(
            int(community_id)
        ).call()
    )

    print(
        f"[MEMBERS] Before={current_count_before}, "
        f"After={current_count_after}"
    )

    return tx_hashes


def store_hash_onchain(
    community_id: int,
    user_address: str,
    version: int,
    top_n: int,
    recommendation_hash
):
    """
    Store recommendation hash on-chain from finalized A_k address.
    """
    trusted_node_address = get_trusted_node_address_for_community(
        community_id
    )

    exists = contract.functions.recordExists(
        int(community_id),
        Web3.to_checksum_address(user_address),
        int(version),
        int(top_n)
    ).call()

    if exists:
        return None, trusted_node_address

    tx_hash = contract.functions.storeRecommendationHash(
        int(community_id),
        Web3.to_checksum_address(user_address),
        int(version),
        int(top_n),
        recommendation_hash
    ).transact(
        {
            "from": trusted_node_address,
            "gas": 500_000
        }
    )

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status != 1:
        raise RuntimeError(
            f"Failed to store hash for community={community_id}, "
            f"user={user_address}, version={version}, topN={top_n}"
        )

    return tx_hash.hex(), trusted_node_address


def normalize_recommendation_columns(recs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Correct Last.fm recommendation column mapping.

    Correct mapping:
        userID   -> user
        artistID -> item
        rank     -> rank

    This avoids duplicate columns by constructing a new DataFrame.
    """

    cols = {
        c.lower().strip(): c
        for c in recs_df.columns
    }

    if "userid" in cols:
        user_col = cols["userid"]
    elif "user_id" in cols:
        user_col = cols["user_id"]
    else:
        raise ValueError(
            "Cannot find original userID column. "
            f"Available columns: {list(recs_df.columns)}"
        )

    if "artistid" in cols:
        item_col = cols["artistid"]
    elif "artist_id" in cols:
        item_col = cols["artist_id"]
    elif "item_id" in cols:
        item_col = cols["item_id"]
    else:
        raise ValueError(
            "Cannot find original artistID/item column. "
            f"Available columns: {list(recs_df.columns)}"
        )

    if "rank" in cols:
        rank_col = cols["rank"]
    elif "rec_rank" in cols:
        rank_col = cols["rec_rank"]
    elif "position" in cols:
        rank_col = cols["position"]
    else:
        raise ValueError(
            "Cannot find rank column. "
            f"Available columns: {list(recs_df.columns)}"
        )

    out = pd.DataFrame({
        "user": recs_df[user_col].astype(int),
        "item": recs_df[item_col].astype(int),
        "rank": recs_df[rank_col].astype(int)
    })

    if "is_positive_in_test" in recs_df.columns:
        out["is_positive_in_test"] = recs_df["is_positive_in_test"]

    print("[COLUMN MAPPING]")
    print("Selected user column:", user_col, "=> user")
    print("Selected item column:", item_col, "=> item")
    print("Selected rank column:", rank_col, "=> rank")
    print("Output columns:", list(out.columns))

    return out


def build_user_topn_recommendations(
    recs_df: pd.DataFrame,
    user_id: int,
    top_n: int
) -> list:
    """
    Build Top-N recommendation list for one original Last.fm userID.
    """

    user_recs = recs_df[
        recs_df["user"].astype(int) == int(user_id)
    ].copy()

    if user_recs.empty:
        return []

    user_recs = user_recs.sort_values("rank")
    user_recs = user_recs[user_recs["rank"] <= int(top_n)]

    recommendations = []

    for _, row in user_recs.iterrows():
        recommendations.append(
            {
                "rank": int(row["rank"]),
                "item": int(row["item"])
            }
        )

    return recommendations


print("=" * 100)
print("Cell 8 — Helper Functions with Corrected Last.fm Mapping")
print("=" * 100)

print("[OK] Helper functions were defined successfully.")
print("[OK] Correct mapping is active: userID => user, artistID => item.")
print("[OK] Cell 8 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 9 — Register A_k and Community Members on RecommendationDelivery Contract
# =====================================================================================

print("=" * 100)
print("Cell 9 — Register A_k and Community Members on RecommendationDelivery Contract")
print("=" * 100)

REGISTER_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "06_register_authorized_nodes_and_members_report.csv"
)

REGISTER_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "06_register_authorized_nodes_and_members_summary.json"
)


def abi_has_function(abi, function_name: str) -> bool:
    return any(
        item.get("type") == "function" and item.get("name") == function_name
        for item in abi
    )


HAS_IS_COMMUNITY_MEMBER = abi_has_function(
    loaded_abi,
    "isCommunityMember"
)

print("Contract has isCommunityMember getter:", HAS_IS_COMMUNITY_MEMBER)


def filter_unregistered_members(
    community_id: int,
    member_addresses: list
) -> list:
    """
    Return only addresses not yet registered on-chain.
    """
    unique_members = sorted(
        set(Web3.to_checksum_address(addr) for addr in member_addresses)
    )

    if not HAS_IS_COMMUNITY_MEMBER:
        return unique_members

    pending_members = []

    for addr in unique_members:
        already_registered = contract.functions.isCommunityMember(
            int(community_id),
            Web3.to_checksum_address(addr)
        ).call()

        if not already_registered:
            pending_members.append(addr)

    return pending_members


registration_rows = []

for community_id in sorted(COMMON_COMMUNITIES):

    print("\n" + "-" * 100)
    print(f"Processing community {community_id}")
    print("-" * 100)

    trusted_node_user_id = get_trusted_node_user_id_for_community(
        community_id
    )

    trusted_node_address = get_trusted_node_address_for_community(
        community_id
    )

    print("Trusted node user_id:", trusted_node_user_id)
    print("Trusted node address:", trusted_node_address)

    trusted_node_address_onchain, ak_tx_hash = ensure_authorized_node(
        community_id
    )

    authorized_node_after = Web3.to_checksum_address(
        contract.functions.authorizedNode(int(community_id)).call()
    )

    ak_matches = (
        authorized_node_after.lower()
        == Web3.to_checksum_address(trusted_node_address).lower()
    )

    if not ak_matches:
        raise RuntimeError(
            f"A_k mismatch after registration for community={community_id}"
        )

    print("[OK] A_k verified on-chain.")

    community_members = members_df[
        members_df["community_id"].astype(int) == int(community_id)
    ].copy()

    if community_members.empty:
        raise RuntimeError(
            f"No members found in members_df for community={community_id}"
        )

    expected_member_user_ids = sorted(
        community_members["user_id"].astype(int).unique().tolist()
    )

    expected_member_addresses = []
    missing_member_addresses = []

    for uid in expected_member_user_ids:
        try:
            expected_member_addresses.append(
                user_address_for_dataset_user(uid)
            )
        except Exception:
            missing_member_addresses.append(uid)

    if missing_member_addresses:
        raise RuntimeError(
            f"Missing blockchain addresses for members in community={community_id}: "
            f"{missing_member_addresses}"
        )

    expected_member_addresses = sorted(
        set(Web3.to_checksum_address(addr) for addr in expected_member_addresses)
    )

    expected_member_count = len(expected_member_addresses)

    print("Expected member count:", expected_member_count)

    current_count_before = int(
        contract.functions.communityMemberCount(int(community_id)).call()
    )

    pending_member_addresses = filter_unregistered_members(
        community_id,
        expected_member_addresses
    )

    pending_member_count = len(pending_member_addresses)

    print("On-chain member count before:", current_count_before)
    print("Pending members to register:", pending_member_count)

    if pending_member_count > 0:
        member_tx_hashes = register_community_members_onchain(
            community_id=community_id,
            member_addresses=pending_member_addresses,
            batch_size=100
        )
    else:
        member_tx_hashes = []
        print("[MEMBERS] No pending members. Registration skipped.")

    current_count_after = int(
        contract.functions.communityMemberCount(int(community_id)).call()
    )

    print("On-chain member count after:", current_count_after)

    if current_count_after != expected_member_count:
        raise RuntimeError(
            f"Member count mismatch for community={community_id}. "
            f"Expected={expected_member_count}, On-chain={current_count_after}"
        )

    print("[OK] Member count verified on-chain.")

    registration_rows.append(
        {
            "community_id": int(community_id),
            "trusted_node_user_id": int(trusted_node_user_id),
            "trusted_node_address": Web3.to_checksum_address(trusted_node_address),
            "authorized_node_onchain": authorized_node_after,
            "ak_matches": bool(ak_matches),
            "ak_registration_tx": ak_tx_hash if ak_tx_hash is not None else "",
            "expected_member_count": int(expected_member_count),
            "onchain_member_count_before": int(current_count_before),
            "pending_member_count_registered": int(pending_member_count),
            "onchain_member_count_after": int(current_count_after),
            "member_count_matches": bool(current_count_after == expected_member_count),
            "member_registration_tx_count": int(len(member_tx_hashes)),
            "member_registration_txs": "|".join(member_tx_hashes),
            "status": "registered_and_verified"
        }
    )


registration_report_df = pd.DataFrame(registration_rows)

registration_report_df.to_csv(
    REGISTER_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "contract_name": "RecommendationDelivery",
    "contract_address": loaded_contract_address,
    "chain_id": int(w3.eth.chain_id),
    "owner_account": OWNER_ACCOUNT,
    "delivery_top_n": int(DELIVERY_TOP_N),
    "delivery_version": int(DELIVERY_VERSION),
    "common_communities": [int(x) for x in sorted(COMMON_COMMUNITIES)],
    "common_communities_count": int(len(COMMON_COMMUNITIES)),
    "registered_communities_count": int(len(registration_report_df)),
    "all_ak_match": bool(registration_report_df["ak_matches"].all()),
    "all_member_counts_match": bool(registration_report_df["member_count_matches"].all()),
    "total_member_registration_transactions": int(
        registration_report_df["member_registration_tx_count"].sum()
    ),
    "report_path": str(REGISTER_REPORT_PATH),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

REGISTER_SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 100)
print("[REGISTRATION REPORT]")
print(registration_report_df.to_string(index=False))

print("\n[OK] Registration report saved to:")
print(REGISTER_REPORT_PATH)

print("\n[OK] Registration summary saved to:")
print(REGISTER_SUMMARY_PATH)

print("\n" + "=" * 100)
print("[FINAL CHECK]")
print("All A_k values were registered and verified:", summary["all_ak_match"])
print("All community member counts were verified:", summary["all_member_counts_match"])
print("Registered communities count:", summary["registered_communities_count"])
print("Total member registration transactions:", summary["total_member_registration_transactions"])
print("=" * 100)

print("\n[OK] Cell 9 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 10B — Re-Diagnose User-ID Alignment After Fix
# =====================================================================================
#
# Purpose:
#   فحص أن userID صار مطابقًا لأعضاء المجتمعات.
#
# Expected:
#   Average coverage ratio قريب من 1.0
#   Communities with zero intersection = []
#
# =====================================================================================

print("=" * 100)
print("Cell 10B — Re-Diagnose User-ID Alignment After Fix")
print("=" * 100)

diagnostic_fixed_rows = []

for rec_file in RECOMMENDATION_FILES:

    community_id = extract_community_id_from_path(rec_file)

    print("\n" + "-" * 100)
    print(f"Re-diagnosing community {community_id}")
    print("-" * 100)

    recs_raw_df = pd.read_csv(rec_file)

    print("Raw columns:", list(recs_raw_df.columns))

    recs_df = normalize_recommendation_columns(recs_raw_df)

    rec_user_ids = set(
        recs_df["user"].astype(int).unique().tolist()
    )

    community_user_ids = set(
        members_df[
            members_df["community_id"].astype(int) == int(community_id)
        ]["user_id"].astype(int).unique().tolist()
    )

    intersection_users = sorted(
        rec_user_ids.intersection(community_user_ids)
    )

    missing_from_recs = sorted(
        community_user_ids - rec_user_ids
    )

    extra_in_recs = sorted(
        rec_user_ids - community_user_ids
    )

    rec_users_count = len(rec_user_ids)
    community_members_count = len(community_user_ids)
    intersection_count = len(intersection_users)

    coverage_ratio = (
        intersection_count / community_members_count
        if community_members_count > 0
        else 0
    )

    rec_min = min(rec_user_ids) if rec_user_ids else None
    rec_max = max(rec_user_ids) if rec_user_ids else None

    comm_min = min(community_user_ids) if community_user_ids else None
    comm_max = max(community_user_ids) if community_user_ids else None

    print("Recommendation users count:", rec_users_count)
    print("Community members count:", community_members_count)
    print("Intersection count:", intersection_count)
    print("Coverage ratio:", round(coverage_ratio, 4))
    print("Recommendation userID range:", rec_min, "to", rec_max)
    print("Community member user_id range:", comm_min, "to", comm_max)
    print("Sample missing community users from recs:", missing_from_recs[:10])
    print("Sample extra rec users not in community:", extra_in_recs[:10])

    diagnostic_fixed_rows.append(
        {
            "community_id": int(community_id),
            "recommendation_users_count": int(rec_users_count),
            "community_members_count": int(community_members_count),
            "intersection_count": int(intersection_count),
            "coverage_ratio": float(coverage_ratio),
            "rec_user_min": rec_min,
            "rec_user_max": rec_max,
            "community_user_min": comm_min,
            "community_user_max": comm_max,
            "missing_from_recs_count": int(len(missing_from_recs)),
            "extra_in_recs_count": int(len(extra_in_recs)),
            "sample_intersection_users": str(intersection_users[:20]),
            "sample_missing_from_recs": str(missing_from_recs[:20]),
            "sample_extra_in_recs": str(extra_in_recs[:20]),
            "recommendation_file": str(rec_file)
        }
    )

diagnostic_fixed_df = pd.DataFrame(diagnostic_fixed_rows)

DIAGNOSTIC_FIXED_OUTPUT_PATH = (
    DELIVERY_OUTPUT_DIR / "08B_user_id_alignment_diagnostic_after_fix.csv"
)

diagnostic_fixed_df.to_csv(
    DIAGNOSTIC_FIXED_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 100)
print("[USER-ID ALIGNMENT DIAGNOSTIC AFTER FIX]")
print(diagnostic_fixed_df.to_string(index=False))

print("\n[OK] Diagnostic after fix saved to:")
print(DIAGNOSTIC_FIXED_OUTPUT_PATH)

print("\n[FINAL CHECK]")
print("Average coverage ratio:", diagnostic_fixed_df["coverage_ratio"].mean())
print("Minimum coverage ratio:", diagnostic_fixed_df["coverage_ratio"].min())
print("Maximum coverage ratio:", diagnostic_fixed_df["coverage_ratio"].max())

print(
    "Communities with zero intersection:",
    diagnostic_fixed_df[
        diagnostic_fixed_df["intersection_count"] == 0
    ]["community_id"].tolist()
)

print(
    "Communities with full coverage:",
    diagnostic_fixed_df[
        diagnostic_fixed_df["coverage_ratio"] == 1.0
    ]["community_id"].tolist()
)

print("\n[OK] Cell 10B completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 10 — Build Payloads and Store Recommendation Hashes On-Chain
# =====================================================================================

print("=" * 100)
print("Cell 10 — Build Payloads and Store Recommendation Hashes On-Chain")
print("=" * 100)

print("Current DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("Current DELIVERY_VERSION:", DELIVERY_VERSION)
print("Current DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)
print("Current PAYLOAD_DIR:", PAYLOAD_DIR)

if int(DELIVERY_VERSION) != 2:
    raise RuntimeError(
        "DELIVERY_VERSION is not 2. النسخة البحثية المعتمدة يجب أن تكون V2."
    )

HASH_STORAGE_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_report.csv"
)

HASH_STORAGE_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_summary.json"
)

PAYLOAD_INDEX_PATH = (
    DELIVERY_OUTPUT_DIR / "07_payload_index.csv"
)


def abi_has_function_local(abi, function_name: str) -> bool:
    return any(
        item.get("type") == "function" and item.get("name") == function_name
        for item in abi
    )


HAS_GET_RECOMMENDATION_HASH = abi_has_function_local(
    loaded_abi,
    "getRecommendationHash"
)

print("Contract has getRecommendationHash:", HAS_GET_RECOMMENDATION_HASH)


def hash_to_hex(h) -> str:
    """
    Convert hash value to 0x-prefixed hex string.
    """
    if isinstance(h, str):
        return h if h.startswith("0x") else "0x" + h

    if isinstance(h, bytes):
        return "0x" + h.hex()

    if hasattr(h, "hex"):
        hx = h.hex()
        return hx if hx.startswith("0x") else "0x" + hx

    raise TypeError(f"Unsupported hash type: {type(h)}")


delivery_rows = []
payload_index_rows = []

total_expected_users = 0
total_payloads_created = 0
total_hashes_stored = 0
total_already_existing = 0
total_skipped_no_recommendations = 0
total_failed = 0

for rec_file in RECOMMENDATION_FILES:

    community_id = extract_community_id_from_path(rec_file)

    print("\n" + "-" * 100)
    print(f"Processing recommendation file for community {community_id}")
    print("-" * 100)
    print("File:", rec_file)

    if community_id not in COMMON_COMMUNITIES:
        print(f"[SKIP] community {community_id} is not in COMMON_COMMUNITIES.")
        continue

    recs_raw_df = pd.read_csv(rec_file)
    recs_df = normalize_recommendation_columns(recs_raw_df)

    print("Rows in recommendation file:", len(recs_df))
    print("Users in recommendation file:", recs_df["user"].nunique())

    community_member_user_ids = sorted(
        members_df[
            members_df["community_id"].astype(int) == int(community_id)
        ]["user_id"].astype(int).unique().tolist()
    )

    if not community_member_user_ids:
        raise RuntimeError(
            f"No community members found for community_id={community_id}"
        )

    total_expected_users += len(community_member_user_ids)

    print("Community members:", len(community_member_user_ids))

    trusted_node_user_id = get_trusted_node_user_id_for_community(
        community_id
    )

    trusted_node_address = get_trusted_node_address_for_community(
        community_id
    )

    print("A_k user_id:", trusted_node_user_id)
    print("A_k address:", trusted_node_address)

    for user_id in community_member_user_ids:

        try:
            user_address = user_address_for_dataset_user(user_id)

            recommendations = build_user_topn_recommendations(
                recs_df=recs_df,
                user_id=user_id,
                top_n=DELIVERY_TOP_N
            )

            if len(recommendations) == 0:
                total_skipped_no_recommendations += 1

                delivery_rows.append(
                    {
                        "community_id": int(community_id),
                        "user_id": int(user_id),
                        "user_address": user_address,
                        "trusted_node_user_id": int(trusted_node_user_id),
                        "trusted_node_address": trusted_node_address,
                        "delivery_version": int(DELIVERY_VERSION),
                        "top_n": int(DELIVERY_TOP_N),
                        "recommendation_count": 0,
                        "payload_path": "",
                        "recommendation_hash": "",
                        "tx_hash": "",
                        "record_exists_before": "",
                        "record_exists_after": "",
                        "stored_hash_matches": "",
                        "status": "skipped_no_recommendations",
                        "error": ""
                    }
                )

                continue

            nonce = (
                f"lastfm|community={community_id}|user={user_id}|"
                f"version={DELIVERY_VERSION}|topN={DELIVERY_TOP_N}"
            )

            payload = build_recommendation_payload_for_hash(
                user_address=user_address,
                community_id=community_id,
                version=DELIVERY_VERSION,
                top_n=DELIVERY_TOP_N,
                recommendations=recommendations,
                nonce=nonce
            )

            payload_json = canonical_json(payload)

            recommendation_hash = compute_recommendation_hash(
                user_address=user_address,
                community_id=community_id,
                version=DELIVERY_VERSION,
                top_n=DELIVERY_TOP_N,
                recommendations=recommendations,
                nonce=nonce
            )

            recommendation_hash_hex = hash_to_hex(recommendation_hash)

            community_payload_dir = PAYLOAD_DIR / f"community_{community_id}"
            community_payload_dir.mkdir(parents=True, exist_ok=True)

            payload_file = (
                community_payload_dir
                / f"user_{user_id}_top{DELIVERY_TOP_N}_v{DELIVERY_VERSION}.json"
            )

            payload_to_save = {
                "payload": payload,
                "canonical_json": payload_json,
                "recommendation_hash": recommendation_hash_hex,
                "trusted_node_user_id": int(trusted_node_user_id),
                "trusted_node_address": trusted_node_address,
                "source_recommendation_file": str(rec_file),
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            payload_file.write_text(
                json.dumps(payload_to_save, indent=2, ensure_ascii=False),
                encoding="utf-8"
            )

            total_payloads_created += 1

            record_exists_before = contract.functions.recordExists(
                int(community_id),
                Web3.to_checksum_address(user_address),
                int(DELIVERY_VERSION),
                int(DELIVERY_TOP_N)
            ).call()

            if record_exists_before:
                tx_hash = ""
                status = "already_exists"
                total_already_existing += 1
            else:
                tx_hash, sender_address = store_hash_onchain(
                    community_id=community_id,
                    user_address=user_address,
                    version=DELIVERY_VERSION,
                    top_n=DELIVERY_TOP_N,
                    recommendation_hash=recommendation_hash
                )

                if tx_hash is None:
                    status = "already_exists"
                    total_already_existing += 1
                    tx_hash = ""
                else:
                    status = "stored_onchain"
                    total_hashes_stored += 1

            record_exists_after = contract.functions.recordExists(
                int(community_id),
                Web3.to_checksum_address(user_address),
                int(DELIVERY_VERSION),
                int(DELIVERY_TOP_N)
            ).call()

            stored_hash_matches = ""

            if HAS_GET_RECOMMENDATION_HASH and record_exists_after:
                stored_hash = contract.functions.getRecommendationHash(
                    int(community_id),
                    Web3.to_checksum_address(user_address),
                    int(DELIVERY_VERSION),
                    int(DELIVERY_TOP_N)
                ).call()

                stored_hash_hex = hash_to_hex(stored_hash)

                stored_hash_matches = (
                    stored_hash_hex.lower() == recommendation_hash_hex.lower()
                )

                if not stored_hash_matches:
                    raise RuntimeError(
                        f"Stored hash mismatch for community={community_id}, user={user_id}"
                    )

            delivery_rows.append(
                {
                    "community_id": int(community_id),
                    "user_id": int(user_id),
                    "user_address": user_address,
                    "trusted_node_user_id": int(trusted_node_user_id),
                    "trusted_node_address": trusted_node_address,
                    "delivery_version": int(DELIVERY_VERSION),
                    "top_n": int(DELIVERY_TOP_N),
                    "recommendation_count": int(len(recommendations)),
                    "payload_path": str(payload_file),
                    "recommendation_hash": recommendation_hash_hex,
                    "tx_hash": tx_hash,
                    "record_exists_before": bool(record_exists_before),
                    "record_exists_after": bool(record_exists_after),
                    "stored_hash_matches": stored_hash_matches,
                    "status": status,
                    "error": ""
                }
            )

            payload_index_rows.append(
                {
                    "community_id": int(community_id),
                    "user_id": int(user_id),
                    "user_address": user_address,
                    "payload_path": str(payload_file),
                    "recommendation_hash": recommendation_hash_hex,
                    "delivery_version": int(DELIVERY_VERSION),
                    "top_n": int(DELIVERY_TOP_N)
                }
            )

        except Exception as e:
            total_failed += 1

            delivery_rows.append(
                {
                    "community_id": int(community_id),
                    "user_id": int(user_id),
                    "user_address": "",
                    "trusted_node_user_id": int(trusted_node_user_id),
                    "trusted_node_address": trusted_node_address,
                    "delivery_version": int(DELIVERY_VERSION),
                    "top_n": int(DELIVERY_TOP_N),
                    "recommendation_count": "",
                    "payload_path": "",
                    "recommendation_hash": "",
                    "tx_hash": "",
                    "record_exists_before": "",
                    "record_exists_after": "",
                    "stored_hash_matches": "",
                    "status": "failed",
                    "error": str(e)
                }
            )

            print(
                f"[ERROR] community={community_id}, user={user_id}, error={e}"
            )

    current_df = pd.DataFrame(delivery_rows)
    current_comm_df = current_df[
        current_df["community_id"].astype(int) == int(community_id)
    ]

    print("[COMMUNITY SUMMARY]")
    print(current_comm_df["status"].value_counts().to_string())


delivery_report_df = pd.DataFrame(delivery_rows)
payload_index_df = pd.DataFrame(payload_index_rows)

delivery_report_df.to_csv(
    HASH_STORAGE_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

payload_index_df.to_csv(
    PAYLOAD_INDEX_PATH,
    index=False,
    encoding="utf-8-sig"
)

status_counts = (
    delivery_report_df["status"]
    .value_counts()
    .to_dict()
    if not delivery_report_df.empty
    else {}
)

successful_records = int(
    delivery_report_df[
        delivery_report_df["status"].isin(
            ["stored_onchain", "already_exists"]
        )
    ].shape[0]
)

failed_records = int(
    delivery_report_df[
        delivery_report_df["status"] == "failed"
    ].shape[0]
)

skipped_no_recs = int(
    delivery_report_df[
        delivery_report_df["status"] == "skipped_no_recommendations"
    ].shape[0]
)

final_summary = {
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "contract_name": "RecommendationDelivery",
    "contract_address": loaded_contract_address,
    "chain_id": int(w3.eth.chain_id),
    "owner_account": OWNER_ACCOUNT,
    "delivery_top_n": int(DELIVERY_TOP_N),
    "delivery_version": int(DELIVERY_VERSION),
    "common_communities": [int(x) for x in sorted(COMMON_COMMUNITIES)],
    "common_communities_count": int(len(COMMON_COMMUNITIES)),
    "recommendation_files_count": int(len(RECOMMENDATION_FILES)),
    "total_expected_community_users": int(total_expected_users),
    "total_payloads_created": int(total_payloads_created),
    "total_hashes_stored_new": int(total_hashes_stored),
    "total_already_existing_records": int(total_already_existing),
    "total_successful_records": int(successful_records),
    "total_skipped_no_recommendations": int(skipped_no_recs),
    "total_failed_records": int(failed_records),
    "status_counts": status_counts,
    "hash_storage_report_path": str(HASH_STORAGE_REPORT_PATH),
    "payload_index_path": str(PAYLOAD_INDEX_PATH),
    "payload_dir": str(PAYLOAD_DIR),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

HASH_STORAGE_SUMMARY_PATH.write_text(
    json.dumps(final_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 100)
print("[HASH STORAGE REPORT]")
print("Report saved to:")
print(HASH_STORAGE_REPORT_PATH)

print("\nPayload index saved to:")
print(PAYLOAD_INDEX_PATH)

print("\nSummary saved to:")
print(HASH_STORAGE_SUMMARY_PATH)

print("\nPayload directory:")
print(PAYLOAD_DIR)

print("\n" + "=" * 100)
print("[FINAL CHECK]")
print("Common communities count:", final_summary["common_communities_count"])
print("Recommendation files count:", final_summary["recommendation_files_count"])
print("Total expected community users:", final_summary["total_expected_community_users"])
print("Total payloads created:", final_summary["total_payloads_created"])
print("Total new hashes stored on-chain:", final_summary["total_hashes_stored_new"])
print("Total already existing records:", final_summary["total_already_existing_records"])
print("Total successful records:", final_summary["total_successful_records"])
print("Total skipped no recommendations:", final_summary["total_skipped_no_recommendations"])
print("Total failed records:", final_summary["total_failed_records"])
print("Status counts:", final_summary["status_counts"])
print("=" * 100)

if final_summary["total_failed_records"] > 0:
    raise RuntimeError(
        "Some recommendation hash storage operations failed. "
        "راجع التقرير لمعرفة التفاصيل."
    )

print("\n[OK] Cell 10 completed.")

# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 11 — Read and Validate Stored Recommendation Hashes
# =====================================================================================

print("=" * 100)
print("Cell 11 — Read and Validate Stored Recommendation Hashes")
print("=" * 100)

print("Current DELIVERY_TOP_N:", DELIVERY_TOP_N)
print("Current DELIVERY_VERSION:", DELIVERY_VERSION)
print("Current DELIVERY_OUTPUT_DIR:", DELIVERY_OUTPUT_DIR)

if int(DELIVERY_VERSION) != 2:
    raise RuntimeError(
        "DELIVERY_VERSION is not 2. هذه الخلية يجب أن تتحقق من النسخة المصححة V2."
    )

HASH_STORAGE_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_report.csv"
)

PAYLOAD_INDEX_PATH = (
    DELIVERY_OUTPUT_DIR / "07_payload_index.csv"
)

HASH_STORAGE_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "07_recommendation_hash_storage_summary.json"
)

VALIDATION_REPORT_PATH = (
    DELIVERY_OUTPUT_DIR / "08_validate_stored_hashes_report.csv"
)

VALIDATION_SUMMARY_PATH = (
    DELIVERY_OUTPUT_DIR / "08_validate_stored_hashes_summary.json"
)

required_files = [
    HASH_STORAGE_REPORT_PATH,
    PAYLOAD_INDEX_PATH,
    HASH_STORAGE_SUMMARY_PATH
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

print("[OK] Required V2 delivery files exist.")
print("Hash storage report:", HASH_STORAGE_REPORT_PATH)
print("Payload index:", PAYLOAD_INDEX_PATH)
print("Hash storage summary:", HASH_STORAGE_SUMMARY_PATH)

hash_report_df = pd.read_csv(HASH_STORAGE_REPORT_PATH)
payload_index_df = pd.read_csv(PAYLOAD_INDEX_PATH)

with open(HASH_STORAGE_SUMMARY_PATH, "r", encoding="utf-8") as f:
    hash_storage_summary = json.load(f)

print("\n[OK] Reports loaded successfully.")
print("Rows in hash storage report:", len(hash_report_df))
print("Rows in payload index:", len(payload_index_df))

stored_rows = hash_report_df[
    hash_report_df["status"].astype(str) == "stored_onchain"
].copy()

skipped_rows = hash_report_df[
    hash_report_df["status"].astype(str) == "skipped_no_recommendations"
].copy()

failed_rows = hash_report_df[
    hash_report_df["status"].astype(str) == "failed"
].copy()

print("\nStatus counts from hash storage report:")
print(hash_report_df["status"].value_counts().to_string())

if not failed_rows.empty:
    raise RuntimeError(
        "Hash storage report contains failed records. لا تكمل التحقق قبل مراجعتها."
    )

if len(payload_index_df) != len(stored_rows):
    raise RuntimeError(
        f"Payload index rows ({len(payload_index_df)}) do not match stored_onchain rows ({len(stored_rows)})."
    )

print("\n[OK] Payload index count matches stored_onchain count.")


def hash_to_hex_v11(h) -> str:
    """
    Convert hash value to 0x-prefixed hex string.
    """
    if isinstance(h, str):
        return h if h.startswith("0x") else "0x" + h

    if isinstance(h, bytes):
        return "0x" + h.hex()

    if hasattr(h, "hex"):
        hx = h.hex()
        return hx if hx.startswith("0x") else "0x" + hx

    raise TypeError(f"Unsupported hash type: {type(h)}")


def recompute_hash_from_payload(payload: dict) -> str:
    """
    Recompute keccak256 hash from payload using canonical JSON.
    """
    msg = canonical_json(payload)
    return hash_to_hex_v11(w3.keccak(text=msg))


validation_rows = []

for idx, row in payload_index_df.iterrows():

    community_id = int(row["community_id"])
    user_id = int(row["user_id"])
    user_address = Web3.to_checksum_address(row["user_address"])
    payload_path = Path(str(row["payload_path"]))
    index_hash = hash_to_hex_v11(str(row["recommendation_hash"]))
    version = int(row["delivery_version"])
    top_n = int(row["top_n"])

    print(f"\nValidating community={community_id}, user_id={user_id}")

    row_status = "validated"
    error_msg = ""

    payload_exists = payload_path.exists()

    payload_hash = ""
    recomputed_hash = ""
    onchain_hash = ""

    payload_hash_matches_index = False
    recomputed_hash_matches_payload = False
    recomputed_hash_matches_index = False
    onchain_record_exists = False
    onchain_hash_matches_recomputed = False
    final_valid = False

    try:
        if not payload_exists:
            raise FileNotFoundError(
                f"Payload file not found: {payload_path}"
            )

        with open(payload_path, "r", encoding="utf-8") as f:
            payload_file_data = json.load(f)

        if "payload" not in payload_file_data:
            raise RuntimeError(
                f"Missing 'payload' key in {payload_path}"
            )

        if "recommendation_hash" not in payload_file_data:
            raise RuntimeError(
                f"Missing 'recommendation_hash' key in {payload_path}"
            )

        payload = payload_file_data["payload"]
        payload_hash = hash_to_hex_v11(
            payload_file_data["recommendation_hash"]
        )

        recomputed_hash = recompute_hash_from_payload(payload)

        payload_hash_matches_index = (
            payload_hash.lower() == index_hash.lower()
        )

        recomputed_hash_matches_payload = (
            recomputed_hash.lower() == payload_hash.lower()
        )

        recomputed_hash_matches_index = (
            recomputed_hash.lower() == index_hash.lower()
        )

        onchain_record_exists = contract.functions.recordExists(
            int(community_id),
            Web3.to_checksum_address(user_address),
            int(version),
            int(top_n)
        ).call()

        if not onchain_record_exists:
            raise RuntimeError(
                f"On-chain record does not exist for community={community_id}, user_id={user_id}"
            )

        onchain_hash_raw = contract.functions.getRecommendationHash(
            int(community_id),
            Web3.to_checksum_address(user_address),
            int(version),
            int(top_n)
        ).call()

        onchain_hash = hash_to_hex_v11(onchain_hash_raw)

        onchain_hash_matches_recomputed = (
            onchain_hash.lower() == recomputed_hash.lower()
        )

        final_valid = (
            payload_exists
            and payload_hash_matches_index
            and recomputed_hash_matches_payload
            and recomputed_hash_matches_index
            and onchain_record_exists
            and onchain_hash_matches_recomputed
        )

        if not final_valid:
            row_status = "invalid"

    except Exception as e:
        row_status = "failed_validation"
        error_msg = str(e)

    validation_rows.append(
        {
            "community_id": int(community_id),
            "user_id": int(user_id),
            "user_address": user_address,
            "delivery_version": int(version),
            "top_n": int(top_n),
            "payload_path": str(payload_path),
            "payload_exists": bool(payload_exists),
            "payload_hash": payload_hash,
            "index_hash": index_hash,
            "recomputed_hash": recomputed_hash,
            "onchain_hash": onchain_hash,
            "payload_hash_matches_index": bool(payload_hash_matches_index),
            "recomputed_hash_matches_payload": bool(recomputed_hash_matches_payload),
            "recomputed_hash_matches_index": bool(recomputed_hash_matches_index),
            "onchain_record_exists": bool(onchain_record_exists),
            "onchain_hash_matches_recomputed": bool(onchain_hash_matches_recomputed),
            "final_valid": bool(final_valid),
            "status": row_status,
            "error": error_msg
        }
    )

    if row_status != "validated":
        print("[WARNING]", row_status, error_msg)


validation_report_df = pd.DataFrame(validation_rows)

validation_report_df.to_csv(
    VALIDATION_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

total_records_checked = int(len(validation_report_df))

total_valid = int(
    validation_report_df["final_valid"].astype(bool).sum()
)

total_invalid = int(
    total_records_checked - total_valid
)

status_counts = (
    validation_report_df["status"].value_counts().to_dict()
    if not validation_report_df.empty
    else {}
)

summary = {
    "dataset": "Last.fm",
    "stage": "Recommendation Delivery",
    "step": "Cell 11 Hash Validation",
    "contract_name": "RecommendationDelivery",
    "contract_address": loaded_contract_address,
    "chain_id": int(w3.eth.chain_id),
    "delivery_top_n": int(DELIVERY_TOP_N),
    "delivery_version": int(DELIVERY_VERSION),
    "common_communities_count": int(len(COMMON_COMMUNITIES)),
    "stored_records_from_cell10": int(len(stored_rows)),
    "skipped_no_recommendations_from_cell10": int(len(skipped_rows)),
    "total_records_checked": int(total_records_checked),
    "total_valid_records": int(total_valid),
    "total_invalid_records": int(total_invalid),
    "all_payload_files_exist": bool(validation_report_df["payload_exists"].all()),
    "all_payload_hashes_match_index": bool(validation_report_df["payload_hash_matches_index"].all()),
    "all_recomputed_hashes_match_payload": bool(validation_report_df["recomputed_hash_matches_payload"].all()),
    "all_recomputed_hashes_match_index": bool(validation_report_df["recomputed_hash_matches_index"].all()),
    "all_onchain_records_exist": bool(validation_report_df["onchain_record_exists"].all()),
    "all_onchain_hashes_match_recomputed": bool(validation_report_df["onchain_hash_matches_recomputed"].all()),
    "status_counts": status_counts,
    "validation_report_path": str(VALIDATION_REPORT_PATH),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

VALIDATION_SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 100)
print("[HASH VALIDATION REPORT]")
print("Validation report saved to:")
print(VALIDATION_REPORT_PATH)

print("\nValidation summary saved to:")
print(VALIDATION_SUMMARY_PATH)

print("\n" + "=" * 100)
print("[FINAL CHECK]")
print("Stored records from Cell 10:", summary["stored_records_from_cell10"])
print("Skipped no recommendations from Cell 10:", summary["skipped_no_recommendations_from_cell10"])
print("Total records checked:", summary["total_records_checked"])
print("Total valid records:", summary["total_valid_records"])
print("Total invalid records:", summary["total_invalid_records"])
print("All payload files exist:", summary["all_payload_files_exist"])
print("All payload hashes match index:", summary["all_payload_hashes_match_index"])
print("All recomputed hashes match payload:", summary["all_recomputed_hashes_match_payload"])
print("All recomputed hashes match index:", summary["all_recomputed_hashes_match_index"])
print("All on-chain records exist:", summary["all_onchain_records_exist"])
print("All on-chain hashes match recomputed:", summary["all_onchain_hashes_match_recomputed"])
print("Status counts:", summary["status_counts"])
print("=" * 100)

if summary["total_invalid_records"] > 0:
    raise RuntimeError(
        "Some stored recommendation hashes failed validation. راجع تقرير Cell 11."
    )

print("\n[OK] Cell 11 completed.")

